# Bill-of-lading classifier: dataset + baseline Transformer
Part 1 of the architecture-variants experiment. Contents: environment info, synthetic
dataset, tokenizer, baseline model, training with seeds 0/1/2, test evaluation.
Every number printed below comes from running this file.

In [ ]:
import json, math, os, platform, random, re, sys, time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## 0. Environment verification (GPU required)
This notebook stops if CUDA is not available. In Colab: Runtime > Change runtime type > GPU.

In [ ]:
# Must be set before the first CUDA call. Used with warn_only determinism below.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

print("Python version         :", sys.version.split()[0])
print("PyTorch version        :", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. This experiment must run on a GPU (Colab: Runtime > Change runtime type > "
        "T4 GPU), then restart and run all cells. It will not fall back to CPU.")

DEVICE = torch.device("cuda")
_p = torch.cuda.get_device_properties(0)
print("GPU name               :", torch.cuda.get_device_name(0))
print("CUDA version (torch)   :", torch.version.cuda)
print("GPU compute capability :", f"{_p.major}.{_p.minor}")
print("GPU memory (GB)        :", round(_p.total_memory / 1e9, 2))

ENV = {"python": sys.version.split()[0], "platform": platform.platform(), "torch": torch.__version__,
       "device": str(DEVICE), "cuda_available": True, "gpu_name": torch.cuda.get_device_name(0),
       "cuda_version": torch.version.cuda, "compute_capability": f"{_p.major}.{_p.minor}",
       "gpu_memory_gb": round(_p.total_memory / 1e9, 2)}

# Best-effort determinism. GPU kernels are not guaranteed bitwise identical to the CPU run
# (or across GPU types), so expect small numeric differences from the CPU reference results.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Synthetic bill-of-lading dataset
Six classes. Two are rare (hazmat, oversize/heavy), 4% each.

**Input corruption (train, val AND test):** OCR character errors, missing fields,
odd spacing, truncated values, and a conflicting instruction borrowed from another class.
The class label is the underlying shipment type and is never changed by the corruption.

**Label noise:** 3% of *training* labels are flipped to a different random class.
Validation and test labels are clean.

In [ ]:
CLASSES = ["general_dry", "retail_pallet", "refrigerated", "fragile_high_value", "hazmat", "oversize_heavy"]
PROPS = [0.34, 0.26, 0.20, 0.12, 0.04, 0.04]
RARE_IDS = [CLASSES.index("hazmat"), CLASSES.index("oversize_heavy")]
N_TOTAL = 15_000
SPLIT = {"train": 0.70, "val": 0.10, "test": 0.20}
DATA_SEED = 42
TRAIN_LABEL_NOISE = 0.03
MAX_LEN = 128

# Corruption settings, fixed before looking at any model results
P_DROP_FIELD = 0.25       # each class-specific field is missing with this probability
P_DROP_COMMODITY = 0.05
P_CONFLICT = 0.15         # an instruction typical of another class is appended
P_TRUNCATE = 0.06         # a field is cut off mid-value
OCR_LEVELS = [0.0, 0.01, 0.03, 0.06]     # per-character error rate, one level per document
SPACE_LEVELS = [0.0, 0.03, 0.10]

NAMES = ["ACME", "NORTHSTAR", "BLUE RIDGE", "PACIFIC", "MIDWEST", "SUMMIT", "HARBOR", "KEYSTONE", "LONE STAR", "ATLAS"]
SUFFIX = ["SUPPLY CO", "DISTRIBUTION", "INDUSTRIES", "FOODS", "LOGISTICS", "MFG", "TRADING", "WHSE"]
CITIES = ["DALLAS TX", "MEMPHIS TN", "RENO NV", "ATLANTA GA", "COLUMBUS OH", "TAMPA FL", "DENVER CO", "NEWARK NJ"]
GENERIC_INSTR = ["CALL BEFORE DELIVERY", "LIFTGATE REQUIRED", "APPT REQUIRED", "DELIVER 8AM-4PM", "DO NOT STACK", "INSIDE DELIVERY"]

CLASS_INSTR = {   # used as the conflicting instruction pool
    0: ["NO SPECIAL HANDLING", "STACKABLE", "DRY VAN"],
    1: ["STORE REPLENISHMENT", "ROUTING GUIDE COMPLIANT", "ASN SENT"],
    2: ["KEEP FROZEN", "DO NOT FREEZE", "REEFER CONTINUOUS"],
    3: ["FRAGILE", "SIGNATURE REQUIRED", "HANDLE WITH CARE", "THIS SIDE UP"],
    4: ["HAZMAT", "PLACARD REQUIRED", "FLAMMABLE"],
    5: ["OVERSIZE LOAD", "ESCORT REQUIRED", "FLATBED TARP"],
}


def _party(r):
    return f"{r.choice(NAMES)} {r.choice(SUFFIX)} {r.choice(CITIES)}"


def make_fields(c, r):
    """Returns (always_present_fields, droppable_class_fields, commodity_field)."""
    base = [f"BOL {r.randint(100000, 999999)}", f"DATE {r.randint(1, 12):02d}/{r.randint(1, 28):02d}/25",
            f"SHIPPER {_party(r)}", f"CONSIGNEE {_party(r)}", f"CARRIER {r.choice(NAMES)} FREIGHT",
            r.choice(["PREPAID", "COLLECT", "3RD PARTY"])]
    sig, com = [], None
    pcs = r.randint(1, 40)
    if c == 0:
        com = "DESC " + r.choice(["PAPER PRODUCTS", "STEEL BRACKETS", "EMPTY PLASTIC BOTTLES", "CARDBOARD CARTONS",
                                  "PVC FITTINGS", "FASTENERS", "CERAMIC TILE", "LUMBER", "TIRES"])
        sig = [f"{pcs} {r.choice(['PLT', 'CTN', 'BDL'])}", f"WT {r.randint(2000, 44000)} LB",
               f"FREIGHT CLASS {r.choice([50, 55, 60, 65, 70, 77.5, 85])}", f"NMFC {r.randint(10000, 160000)}",
               r.choice(CLASS_INSTR[0])]
    elif c == 1:
        com = "DESC " + r.choice(["APPAREL", "HOUSEWARES", "SMALL APPLIANCES", "TOYS", "HOME TEXTILES", "PET SUPPLIES"])
        sig = [f"{pcs} PLT", f"WT {r.randint(2000, 30000)} LB", f"PO {r.randint(10000000, 99999999)}",
               f"DC {r.randint(1, 99):02d}", r.choice(["SSCC LABELS APPLIED", "PALLET LABELS ATTACHED"]),
               r.choice(CLASS_INSTR[1]), "DELIVERY APPT REQUIRED"]
    elif c == 2:
        frozen = r.random() < 0.5
        com = "DESC " + r.choice(["FROZEN POULTRY", "ICE CREAM", "FROZEN SEAFOOD"] if frozen
                                 else ["FRESH PRODUCE", "DAIRY", "CHILLED MEAT", "FLORAL"])
        sig = [f"{pcs} PLT", f"WT {r.randint(8000, 42000)} LB",
               f"SET TEMP {r.choice([-20, -18, -15]) if frozen else r.choice([34, 36, 38])} {'C' if frozen else 'F'}",
               "REEFER", r.choice(CLASS_INSTR[2])]
    elif c == 3:
        com = "DESC " + r.choice(["LCD MONITORS", "GLASSWARE", "LAPTOPS", "FRAMED ARTWORK", "MEDICAL DEVICES", "JEWELRY"])
        sig = [f"{pcs} CTN", f"WT {r.randint(200, 9000)} LB", f"DECLARED VALUE ${r.randint(20, 900) * 1000}",
               r.choice(CLASS_INSTR[3]), r.choice(CLASS_INSTR[3])]
    elif c == 4:
        com = "DESC " + r.choice(["GASOLINE", "FLAMMABLE LIQUID NOS", "PAINT", "LITHIUM ION BATTERIES", "ETHANOL SOLUTION",
                                  "NITRIC ACID", "SODIUM HYDROXIDE SOLUTION", "AEROSOLS"])
        sig = [f"{pcs} DRM", f"WT {r.randint(500, 30000)} LB", f"UN{r.choice([1203, 1993, 1263, 3480, 1170, 2031, 1824, 1950])}",
               f"CLASS {r.choice(['3', '8', '9', '2.1', '4.1'])}", f"PG {r.choice(['II', 'III'])}",
               "PLACARD REQUIRED", f"24HR EMERGENCY CONTACT {r.randint(200, 999)}-{r.randint(200, 999)}-{r.randint(1000, 9999)}"]
    else:
        com = "DESC " + r.choice(["EXCAVATOR", "CNC MACHINE", "TRANSFORMER", "WIND BLADE SECTION", "INDUSTRIAL BOILER",
                                  "STEEL BEAMS", "PRECAST CONCRETE"])
        sig = ["1 PC", f"WT {r.randint(42000, 140000)} LB", f"DIMS {r.randint(240, 600)}X{r.randint(96, 160)}X{r.randint(96, 170)} IN",
               "OVERSIZE LOAD", f"PERMIT {r.randint(100000, 999999)}", "ESCORT REQUIRED", "FLATBED TARP"]
    return base, sig, com


OCR_MAP = {"O": "0", "0": "O", "I": "1", "1": "I", "S": "5", "5": "S", "B": "8", "8": "B", "E": "F", "M": "N", "L": "I"}


def ocr_corrupt(s, p, r):
    out = []
    for ch in s:
        u = r.random()
        if ch in OCR_MAP and u < p:
            out.append(OCR_MAP[ch])
        elif ch.isalnum() and u < 1.5 * p:      # dropped character
            continue
        else:
            out.append(ch)
    return "".join(out)


def space_corrupt(s, p, r):
    out = []
    for ch in s:
        if ch == " " and r.random() < p:
            out.append(r.choice(["  ", "   ", ""]))
        elif ch.isalpha() and r.random() < p / 4:
            out.append(ch + " ")
        else:
            out.append(ch)
    return "".join(out)


def make_doc(c, r):
    base, sig, com = make_fields(c, r)
    sig = [f for f in sig if r.random() > P_DROP_FIELD]
    if com is not None and r.random() > P_DROP_COMMODITY:
        sig.insert(0, com)
    extra = [r.choice(GENERIC_INSTR)] if r.random() < 0.5 else []
    if r.random() < P_CONFLICT:
        other = r.choice([k for k in range(len(CLASSES)) if k != c])
        extra.append(r.choice(CLASS_INSTR[other]))
    fields = base + sig + extra
    # keep header fields first, shuffle the rest lightly so order is not a perfect cue
    head, tail = fields[:6], fields[6:]
    r.shuffle(tail)
    fields = head + tail
    fields = [f[: max(3, int(len(f) * r.uniform(0.3, 0.9)))] if r.random() < P_TRUNCATE else f for f in fields]
    text = r.choice([" ", " | ", " ; ", "\n"]).join(fields)
    text = ocr_corrupt(text, r.choice(OCR_LEVELS), r)
    return space_corrupt(text, r.choice(SPACE_LEVELS), r)


def build_dataset(seed=DATA_SEED):
    r = random.Random(seed)
    counts = [round(p * N_TOTAL) for p in PROPS]
    splits = {k: ([], []) for k in SPLIT}
    for c, n in enumerate(counts):
        n_tr, n_va = round(n * SPLIT["train"]), round(n * SPLIT["val"])
        bounds = {"train": (0, n_tr), "val": (n_tr, n_tr + n_va), "test": (n_tr + n_va, n)}
        for k, (a, b) in bounds.items():
            for _ in range(b - a):
                splits[k][0].append(make_doc(c, r))
                splits[k][1].append(c)
    data = {}
    for k, (texts, labels) in splits.items():
        idx = list(range(len(texts)))
        r.shuffle(idx)
        data[k] = ([texts[i] for i in idx], np.array([labels[i] for i in idx]))
    # label noise on TRAIN labels only
    tr_texts, y_clean = data["train"]
    y_noisy = y_clean.copy()
    nr = np.random.RandomState(seed)
    flip = nr.rand(len(y_noisy)) < TRAIN_LABEL_NOISE
    for i in np.where(flip)[0]:
        y_noisy[i] = nr.choice([k for k in range(len(CLASSES)) if k != y_clean[i]])
    data["train"] = (tr_texts, y_noisy)
    return data, y_clean, flip


data, y_train_clean, flipped = build_dataset()

# Fingerprint of the exact dataset (texts + labels, all splits). If this differs from the reference
# below, the dataset on this machine is NOT the one used in the earlier CPU run.
import hashlib
_h = hashlib.sha256()
for _k in ("train", "val", "test"):
    _h.update(json.dumps([data[_k][0], data[_k][1].tolist()]).encode())
DATA_SHA256 = _h.hexdigest()
REFERENCE_DATA_SHA256 = "e74c4242ce78d13fec8ad56ebe0647c6598766c815068a00ccb2369c6307d610"  # from the earlier CPU run
REFERENCE_VOCAB = 5565
print("dataset sha256:", DATA_SHA256)
if DATA_SHA256 != REFERENCE_DATA_SHA256:
    raise RuntimeError("Dataset differs from the reference (data seed 42) dataset. Stopping so a different "
                       "experiment is not run by accident.")
print("dataset matches the reference dataset: OK")
print("Example documents:")
for c in range(len(CLASSES)):
    i = int(np.where(data["test"][1] == c)[0][0])
    print(f"[{CLASSES[c]}]", data["test"][0][i].replace("\n", " / ")[:230])
print()
dist = {k: np.bincount(v[1], minlength=6).tolist() for k, v in data.items()}
print("class counts per split", dist)
print(f"train label noise: {int(flipped.sum())} of {len(flipped)} labels flipped ({flipped.mean():.2%}); val/test labels are clean")

## 2. Tokenizer
Lowercase, split into alphanumeric runs and single punctuation marks. Vocabulary is built from the
training texts only (min frequency 2). Fixed length 128, pad id 0, unknown id 1.

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+|[^\sa-z0-9]")
PAD, UNK = 0, 1


def tok(s):
    return TOKEN_RE.findall(s.lower())


counter = Counter(t for s in data["train"][0] for t in tok(s))
itos = ["<pad>", "<unk>"] + [t for t, n in counter.most_common() if n >= 2]
stoi = {t: i for i, t in enumerate(itos)}
VOCAB = len(itos)


def encode(texts):
    ids = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
    lens, trunc = [], 0
    for i, s in enumerate(texts):
        t = [stoi.get(w, UNK) for w in tok(s)]
        trunc += len(t) > MAX_LEN
        t = t[:MAX_LEN]
        ids[i, : len(t)] = t
        lens.append(len(t))
    return torch.from_numpy(ids), np.array(lens), trunc


X, Y, TOKSTATS = {}, {}, {}
for k, (texts, labels) in data.items():
    ids, lens, trunc = encode(texts)
    X[k], Y[k] = ids, torch.from_numpy(labels)
    TOKSTATS[k] = {"n_docs": len(texts), "mean_len": float(lens.mean()), "max_len_before_trunc_cap": int(lens.max()),
                   "pct_truncated": 100 * trunc / len(texts), "nonpad_tokens": int(lens.sum())}
unk_rate = {k: float((X[k] == UNK).sum() / (X[k] != PAD).sum()) for k in X}
print("vocab size:", VOCAB)
assert VOCAB == REFERENCE_VOCAB, f"vocab size {VOCAB} != reference {REFERENCE_VOCAB}"
print("token stats:", json.dumps(TOKSTATS, indent=1))
print("unk rate:", {k: round(v, 4) for k, v in unk_rate.items()})

## 3. Baseline model
Embedding + learned positions, 4 pre-LN Transformer blocks (d=128, 4 heads, FFN 256),
padding-aware softmax attention, mean pooling over non-padding tokens, linear head.
The attention is its own module so a later variant can swap it out.

In [ ]:
D_MODEL, N_LAYERS, N_HEADS, D_FF = 128, 4, 4, 256


class SoftmaxSelfAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)

    def forward(self, x, pad_mask):          # pad_mask: (B, T) True where real token
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        key_mask = pad_mask[:, None, None, :]                       # padded keys are never attended to
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
        return self.out(o.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, attn):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.attn = attn
        self.ff = nn.Sequential(nn.Linear(D_MODEL, D_FF), nn.GELU(), nn.Linear(D_FF, D_MODEL))

    def forward(self, x, pad_mask):
        x = x + self.attn(self.ln1(x), pad_mask)
        return x + self.ff(self.ln2(x))


class Classifier(nn.Module):
    def __init__(self, vocab, n_classes, attn_factory):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, D_MODEL, padding_idx=PAD)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)
        self.blocks = nn.ModuleList([Block(attn_factory()) for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, n_classes)

    def forward(self, ids):
        pad_mask = ids != PAD
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(ids.shape[1], device=ids.device))
        for b in self.blocks:
            x = b(x, pad_mask)
        x = self.ln_f(x)
        m = pad_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1)               # mean over real tokens only
        return self.head(pooled)


def build_baseline():
    return Classifier(VOCAB, len(CLASSES), lambda: SoftmaxSelfAttention(D_MODEL, N_HEADS))


def param_counts(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.tok_emb.weight.numel() + model.pos_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# Sanity check: padding must not change the output. Logits with the full 128 positions should match
# logits when the trailing all-padding columns are cut off (so padding is masked in attention and pooling).
_m = build_baseline().eval()
_ids = X["val"][:8]
_L = int((_ids != PAD).sum(1).max())
with torch.no_grad():
    _diff = (_m(_ids) - _m(_ids[:, :_L])).abs().max().item()
print(f"padding invariance: max |logit diff| = {_diff:.2e} (cut to {_L} of {MAX_LEN} positions)")
assert _diff < 1e-4, "padding is leaking into the output"
print("param counts:", param_counts(_m))

## 4. Training
AdamW, lr 1e-3, weight decay 0.01, 100 warmup steps then linear decay to 0, batch 64, 8 epochs,
plain cross-entropy (no class weights, no resampling), fp32. The model after the last epoch is the model
that gets evaluated. Validation is only monitored. Test is evaluated once, after training.

In [ ]:
EPOCHS, BATCH, LR, WD, WARMUP = 8, 64, 1e-3, 0.01, 100
SEEDS = [0, 1, 2]


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


@torch.no_grad()
def predict(model, ids, bs=256):
    model.eval()
    return torch.cat([model(ids[i:i + bs].to(DEVICE)).argmax(-1).cpu() for i in range(0, len(ids), bs)]).numpy()


def metrics(y_true, y_pred):
    per_class = {}
    for c, name in enumerate(CLASSES):
        m = y_true == c
        per_class[name] = float((y_pred[m] == c).mean())      # per-class accuracy = recall of that class
    rare = [per_class[CLASSES[i]] for i in RARE_IDS]
    return {"overall_acc": float((y_true == y_pred).mean()), "per_class_acc": per_class,
            "rare_class_recall_macro": float(np.mean(rare)),
            "rare_support": {CLASSES[i]: int((y_true == i).sum()) for i in RARE_IDS}}


def train_baseline(seed):
    set_seed(seed)
    model = build_baseline().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    n = len(X["train"])
    steps_per_epoch = math.ceil(n / BATCH)
    total = EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / WARMUP if s < WARMUP else max(0.0, (total - s) / (total - WARMUP)))
    g = torch.Generator().manual_seed(seed)
    xtr, ytr = X["train"], Y["train"]
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    log, t0 = [], time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, generator=g)
        run, cnt = 0.0, 0
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            xb, yb = xtr[idx].to(DEVICE), ytr[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step(); sched.step()
            run += loss.item() * len(idx); cnt += len(idx)
        val_acc = float((predict(model, X["val"]) == Y["val"].numpy()).mean())
        log.append({"epoch": ep + 1, "train_loss": run / cnt, "val_acc": val_acc})
        print(f"  seed {seed} epoch {ep + 1}/{EPOCHS} train_loss {run / cnt:.4f} val_acc {val_acc:.4f}")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return model, log, time.time() - t0

In [ ]:
results = {"env": ENV, "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "wd": WD, "warmup": WARMUP,
                                  "max_len": MAX_LEN, "d_model": D_MODEL, "layers": N_LAYERS, "heads": N_HEADS,
                                  "d_ff": D_FF, "vocab": VOCAB, "train_label_noise": TRAIN_LABEL_NOISE,
                                  "data_seed": DATA_SEED, "data_sha256": DATA_SHA256},
           "class_counts": dist, "token_stats": TOKSTATS, "baseline": {}}
y_test = Y["test"].numpy()
for seed in SEEDS:
    model, log, secs = train_baseline(seed)
    pred = predict(model, X["test"])
    r = metrics(y_test, pred)
    r.update(train_seconds=secs, epoch_log=log, params=param_counts(model),
             val_acc_final=log[-1]["val_acc"])
    results["baseline"][f"seed{seed}"] = r
    print(f"seed {seed}: test acc {r['overall_acc']:.4f} | rare macro recall {r['rare_class_recall_macro']:.4f} | {secs:.0f}s")
    torch.save(model.state_dict(), f"{OUT_DIR}/baseline_seed{seed}.pt")

## 5. Baseline report (seeds 0, 1, 2)

In [ ]:
import pandas as pd

HAZ, OVR = "hazmat", "oversize_heavy"
runs = results["baseline"]
print("GPU:", ENV["gpu_name"], "| CUDA", ENV["cuda_version"], "| cc", ENV["compute_capability"])

per_seed = pd.DataFrame({
    sd: {"train_seconds": r["train_seconds"], "final_val_acc": r["val_acc_final"],
         "overall_test_acc": r["overall_acc"],
         **{f"acc_{c}": r["per_class_acc"][c] for c in CLASSES},
         "rare_macro_recall": r["rare_class_recall_macro"]}
    for sd, r in runs.items()})
print(per_seed.round(4).to_string())

p = runs["seed0"]["params"]
print(f"\nparams: total {p['total']:,} | embedding {p['embedding']:,} | non-embedding {p['non_embedding']:,}")

agg = pd.DataFrame({"overall_test_acc": per_seed.loc["overall_test_acc"],
                    "hazmat_recall": per_seed.loc[f"acc_{HAZ}"],
                    "oversize_recall": per_seed.loc[f"acc_{OVR}"],
                    "rare_macro_recall": per_seed.loc["rare_macro_recall"]}).T
agg["mean"], agg["std"] = agg.mean(axis=1), agg.std(axis=1, ddof=1)
print("\nmean and sample std (ddof=1) over 3 seeds")
print(agg[["mean", "std"]].round(4).to_string())

results["summary"] = {"mean": agg["mean"].to_dict(), "std": agg["std"].to_dict()}
with open(f"{OUT_DIR}/baseline_results_cuda.json", "w") as f:
    json.dump(results, f, indent=1)

# Part 2: Linear-attention variant
Only the sequence-mixing layer changes. `Classifier`, `Block`, the dataset, tokenizer, optimizer,
schedule and seeds are the same objects the baseline used. The baseline cells above are not re-run here.

**How to run:** run this section in the same Colab session after the baseline cells. If the runtime
was restarted, run the setup cells (0 to the model definitions) but skip the baseline training cell;
the comparison cell then reads `results/baseline_results_cuda.json`.

**Methodology limitation (kept from Part 1):** CUDA kernels are not guaranteed to be bitwise
deterministic. Determinism is requested with `warn_only=True`. In the baseline run on the T4, PyTorch warned that
"Memory Efficient attention defaults to a non-deterministic algorithm" (softmax attention's backward pass).
The baseline was not changed or rerun because of this. The linear-attention block uses `einsum` and
elementwise ops instead of that kernel, so the warning may not appear for it. The two models therefore
may not have the same degree of run-to-run GPU nondeterminism, and a different GPU can shift numbers slightly.
Fixed seeds, the identical dataset, and identical initial weights per seed reduce this variation but do not remove it.
Latency is not measured in this section, and no speed claim is made for linear attention.


In [ ]:
# --- Linear attention: phi(x) = ELU(x) + 1, non-causal, padding-aware -----------------------
class LinearSelfAttention(nn.Module):
    """Same parameters as SoftmaxSelfAttention (qkv and out projections), different mixing.

    out_i = phi(q_i)^T (sum_j phi(k_j) v_j^T) / (phi(q_i)^T sum_j phi(k_j) + eps)
    Padded keys/values are zeroed before the sums, so they contribute nothing. No T x T matrix is built:
    the sums are (dh x dh) per head.
    """
    def __init__(self, d, h, eps=1e-6):
        super().__init__()
        self.h, self.dh, self.eps = h, d // h, eps
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)

    def forward(self, x, pad_mask):          # x: (B, T, D), pad_mask: (B, T) True = real token
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)   # (B, h, T, dh)
        q, k = F.elu(q) + 1, F.elu(k) + 1
        m = pad_mask[:, None, :, None].to(k.dtype)                                     # (B, 1, T, 1)
        k, v = k * m, v * m                                                            # padded keys/values -> 0
        kv = torch.einsum("bhtd,bhte->bhde", k, v)                                     # sum_j phi(k_j) v_j^T
        z = k.sum(dim=2)                                                               # sum_j phi(k_j)
        num = torch.einsum("bhtd,bhde->bhte", q, kv)
        den = torch.einsum("bhtd,bhd->bht", q, z).unsqueeze(-1) + self.eps
        o = (num / den).transpose(1, 2).reshape(B, T, D)
        return self.out(o)


def build_linear():
    return Classifier(VOCAB, len(CLASSES), lambda: LinearSelfAttention(D_MODEL, N_HEADS))

In [ ]:
# --- Parameter comparison (stop if more than 2% apart) ---------------------------------------
set_seed(0); _base_m = build_baseline()
set_seed(0); _lin_m = build_linear()
pb, pl = param_counts(_base_m), param_counts(_lin_m)
diff = abs(pl["total"] - pb["total"])
pct = 100 * diff / pb["total"]
print(f"Baseline total parameters         : {pb['total']:,}")
print(f"Linear-attention total parameters : {pl['total']:,}")
print(f"Absolute difference               : {diff:,}")
print(f"Percentage difference             : {pct:.4f}%")
if pct > 2.0:
    raise RuntimeError(f"Parameter counts differ by {pct:.2f}% (> 2%). Stopping.")
print("Embedding / non-embedding (linear):", pl["embedding"], "/", pl["non_embedding"])

# Same seed + same parameter shapes/order -> identical initial weights, so seed s starts from the same point
_sb, _sl = _base_m.state_dict(), _lin_m.state_dict()
INIT_IDENTICAL = list(_sb) == list(_sl) and all(torch.equal(_sb[k], _sl[k]) for k in _sb)
print("Initial weights identical to baseline for the same seed:", INIT_IDENTICAL)

In [ ]:
# --- Correctness tests (run on DEVICE, before training) -------------------------------------
def naive_linear_attention(attn, x, pad_mask):
    """Reference only: explicit T x T weights with the same math. Used to check the fast version."""
    B, T, D = x.shape
    q, k, v = attn.qkv(x).view(B, T, 3, attn.h, attn.dh).permute(2, 0, 3, 1, 4)
    q, k = F.elu(q) + 1, F.elu(k) + 1
    A = torch.einsum("bhtd,bhsd->bhts", q, k) * pad_mask[:, None, None, :].to(q.dtype)
    A = A / (A.sum(-1, keepdim=True) + attn.eps)
    return attn.out((A @ v).transpose(1, 2).reshape(B, T, D))


@torch.no_grad()
def padding_invariance(model, ids):
    """Max |logit diff| between the full 128-wide batch and versions with trailing padding removed."""
    model.eval()
    ids = ids.to(DEVICE)
    L = int((ids != PAD).sum(1).max())
    full = model(ids)
    return max((full - model(ids[:, :n])).abs().max().item() for n in (L, min(L + 20, MAX_LEN)))


@torch.no_grad()
def attention_matches_reference(attn, ids):
    attn.eval()
    ids = ids.to(DEVICE)
    mask = ids != PAD
    x = torch.randn(ids.shape[0], ids.shape[1], D_MODEL, device=DEVICE)
    return (attn(x, mask) - naive_linear_attention(attn, x, mask)).abs().max().item()


_ids = X["val"][:64]
_lin_m = _lin_m.to(DEVICE)
PAD_TEST_UNTRAINED = padding_invariance(_lin_m, _ids)
REF_TEST = attention_matches_reference(_lin_m.blocks[0].attn, _ids)
print(f"linear padding invariance (untrained): max |logit diff| = {PAD_TEST_UNTRAINED:.2e}")
print(f"fast linear attention vs explicit T x T reference: max |diff| = {REF_TEST:.2e}")
assert PAD_TEST_UNTRAINED < 1e-4, "padding is leaking into the linear-attention output"
assert REF_TEST < 1e-4, "fast linear attention does not match the reference computation"

# Negative control: the same attention with key/value masking switched off must FAIL the padding test.
# This shows the test can detect padding leakage (an exact 0 above is not a blind test).
class _LeakyLinear(LinearSelfAttention):
    def forward(self, x, pad_mask):
        return super().forward(x, torch.ones_like(pad_mask))


_leaky = Classifier(VOCAB, len(CLASSES), lambda: _LeakyLinear(D_MODEL, N_HEADS)).to(DEVICE)
LEAK_CONTROL = padding_invariance(_leaky, _ids)
print(f"negative control (masking off): max |logit diff| = {LEAK_CONTROL:.2e}  (must be > 1e-4)")
assert LEAK_CONTROL > 1e-4, "padding test is too weak to detect leakage"
del _base_m, _lin_m, _sb, _sl, _leaky

## Training the linear-attention model (seeds 0, 1, 2)
`train_variant` is `train_baseline` with the model constructor passed in. Everything else is identical:
AdamW, lr 1e-3, weight decay 0.01, 100 warm-up steps then linear decay, batch 64, 8 epochs, fp32,
plain cross-entropy, noisy training labels, final-epoch model, test evaluated once after training.

In [ ]:
def train_variant(seed, build_fn):
    set_seed(seed)
    model = build_fn().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    n = len(X["train"])
    steps_per_epoch = math.ceil(n / BATCH)
    total = EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / WARMUP if s < WARMUP else max(0.0, (total - s) / (total - WARMUP)))
    g = torch.Generator().manual_seed(seed)
    xtr, ytr = X["train"], Y["train"]
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    log, t0 = [], time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, generator=g)
        run, cnt = 0.0, 0
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            xb, yb = xtr[idx].to(DEVICE), ytr[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step(); sched.step()
            run += loss.item() * len(idx); cnt += len(idx)
        val_acc = float((predict(model, X["val"]) == Y["val"].numpy()).mean())
        log.append({"epoch": ep + 1, "train_loss": run / cnt, "val_acc": val_acc})
        print(f"  seed {seed} epoch {ep + 1}/{EPOCHS} train_loss {run / cnt:.4f} val_acc {val_acc:.4f}")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return model, log, time.time() - t0

In [ ]:
TEST_EVAL_CALLS = 0
lin = {"env": ENV, "architecture": "linear_attention_elu_plus_1",
       "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "wd": WD, "warmup": WARMUP,
                  "max_len": MAX_LEN, "d_model": D_MODEL, "layers": N_LAYERS, "heads": N_HEADS,
                  "d_ff": D_FF, "vocab": VOCAB, "train_label_noise": TRAIN_LABEL_NOISE,
                  "data_seed": DATA_SEED, "data_sha256": DATA_SHA256},
       "param_check": {"baseline_total": pb["total"], "linear_total": pl["total"], "pct_diff": pct,
                       "init_identical_to_baseline": INIT_IDENTICAL},
       "padding_test_untrained_max_logit_diff": PAD_TEST_UNTRAINED, "leak_control_max_logit_diff": LEAK_CONTROL, "reference_test_max_diff": REF_TEST,
       "linear": {}}
y_test = Y["test"].numpy()
for seed in SEEDS:
    model, log, secs = train_variant(seed, build_linear)
    TEST_EVAL_CALLS += 1                              # test is touched once per model, after training
    pred = predict(model, X["test"])
    r = metrics(y_test, pred)
    r.update(train_seconds=secs, epoch_log=log, params=param_counts(model), val_acc_final=log[-1]["val_acc"],
             padding_test_trained_max_logit_diff=padding_invariance(model, X["val"][:64]))
    lin["linear"][f"seed{seed}"] = r
    print(f"seed {seed}: test acc {r['overall_acc']:.4f} | rare macro recall {r['rare_class_recall_macro']:.4f} | "
          f"{secs:.0f}s | trained-model padding diff {r['padding_test_trained_max_logit_diff']:.2e}")
    torch.save(model.state_dict(), f"{OUT_DIR}/linear_seed{seed}.pt")     # kept for later benchmarking
    del model

with open(f"{OUT_DIR}/linear_results_cuda.json", "w") as f:
    json.dump(lin, f, indent=1)
print("saved:", [f"{OUT_DIR}/linear_seed{s}.pt" for s in SEEDS], "and linear_results_cuda.json")
print("Download these from Colab (or copy to Drive) before the runtime is recycled.")

## Linear-attention report, and comparison with the baseline

In [ ]:
import pandas as pd

if "results" in globals() and len(results.get("baseline", {})) == len(SEEDS):
    base = results
else:
    with open(f"{OUT_DIR}/baseline_results_cuda.json") as f:
        base = json.load(f)
B, L = base["baseline"], lin["linear"]
HAZ, OVR = "hazmat", "oversize_heavy"
print("GPU:", ENV["gpu_name"], "| CUDA", ENV["cuda_version"], "| cc", ENV["compute_capability"])


def per_seed_table(runs):
    return pd.DataFrame({
        sd: {"train_seconds": r["train_seconds"], "final_val_acc": r["val_acc_final"],
             "overall_test_acc": r["overall_acc"],
             **{f"acc_{c}": r["per_class_acc"][c] for c in CLASSES},
             "rare_macro_recall": r["rare_class_recall_macro"]}
        for sd, r in runs.items()})


tl = per_seed_table(L)
print("\nLinear attention, per seed")
print(tl.round(4).to_string())

ROWS = [("overall_test_acc", "overall_test_acc"), ("hazmat_recall", f"acc_{HAZ}"),
        ("oversize_recall", f"acc_{OVR}"), ("rare_macro_recall", "rare_macro_recall")]
tb = per_seed_table(B)
side = pd.DataFrame({name: {"baseline_mean": tb.loc[key].mean(), "baseline_std": tb.loc[key].std(ddof=1),
                            "linear_mean": tl.loc[key].mean(), "linear_std": tl.loc[key].std(ddof=1)}
                     for name, key in ROWS}).T
side["linear_minus_baseline"] = side["linear_mean"] - side["baseline_mean"]
print("\nBaseline vs linear attention: mean and sample std (ddof=1) over seeds 0, 1, 2")
print(side.round(4).to_string())

pc = pd.DataFrame({"baseline_mean": tb.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1),
                   "linear_mean": tl.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1)})
pc["linear_minus_baseline"] = pc["linear_mean"] - pc["baseline_mean"]
print("\nPer-class test accuracy, mean over seeds")
print(pc.round(4).to_string())
print("\nTraining seconds per seed  baseline:", [round(B[s]["train_seconds"]) for s in B],
      "| linear:", [round(L[s]["train_seconds"]) for s in L])
print("(training wall-clock on a shared Colab GPU; this is not an inference-latency measurement)")
print("Seed-to-seed std is on 3 runs and test has 120 documents per rare class: read small gaps with care.")

## Verification checks

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok), detail))

bcfg, lcfg = base["config"], lin["config"]

# 1. same dataset
check("Dataset SHA-256: baseline run == linear run == reference",
      bcfg["data_sha256"] == lcfg["data_sha256"] == DATA_SHA256 == REFERENCE_DATA_SHA256, DATA_SHA256[:16] + "...")
check("Vocab size, class counts and token counts match the baseline run",
      bcfg["vocab"] == VOCAB and base["class_counts"] == dist
      and base["token_stats"]["test"]["nonpad_tokens"] == TOKSTATS["test"]["nonpad_tokens"],
      f"vocab {VOCAB}, test nonpad tokens {TOKSTATS['test']['nonpad_tokens']}")

# 2. same test examples
check("Same test examples (same data hash; identical test class supports in every run)",
      all(B[s]["rare_support"] == L[s]["rare_support"] for s in L) and base["class_counts"]["test"] == dist["test"]
      and len(Y["test"]) == sum(dist["test"]), f"{len(Y['test'])} test docs, rare support {L['seed0']['rare_support']}")

# 3. sequence length
check("Both used sequence length 128", bcfg["max_len"] == lcfg["max_len"] == MAX_LEN == 128
      and all(X[k].shape[1] == 128 for k in X), str({k: tuple(X[k].shape) for k in X}))

# 4. identical training settings
keys = ["epochs", "batch", "lr", "wd", "warmup", "max_len", "d_model", "layers", "heads", "d_ff", "vocab",
        "train_label_noise", "data_seed"]
check("Identical training and architecture settings (all config fields)",
      all(bcfg[k] == lcfg[k] for k in keys) and set(B) == set(L) == {f"seed{s}" for s in SEEDS},
      ", ".join(f"{k}={lcfg[k]}" for k in keys[:6]))
check("Same initial weights as the baseline for each seed", INIT_IDENTICAL)

# 5. padding excluded
base_pad = None
try:
    _bm = build_baseline().to(DEVICE)
    _bm.load_state_dict(torch.load(f"{OUT_DIR}/baseline_seed0.pt", map_location=DEVICE))
    base_pad = padding_invariance(_bm, X["val"][:64]); del _bm
except FileNotFoundError:
    pass
lin_pad = max(r["padding_test_trained_max_logit_diff"] for r in L.values())
check("Padding excluded: linear (trained) logit change < 1e-4 when trailing padding changes",
      lin_pad < 1e-4 and PAD_TEST_UNTRAINED < 1e-4, f"trained max {lin_pad:.2e}, untrained {PAD_TEST_UNTRAINED:.2e}")
check("Fast linear attention matches the explicit masked T x T reference", REF_TEST < 1e-4, f"{REF_TEST:.2e}")
check("Padding test detects leakage (negative control with masking off fails the test)", LEAK_CONTROL > 1e-4,
      f"control diff {LEAK_CONTROL:.2e}")
check("Padding excluded: baseline (trained seed 0) logit change < 1e-4",
      base_pad is not None and base_pad < 1e-4, "checkpoint not found" if base_pad is None else f"{base_pad:.2e}")

# 6. val/test labels are clean: regenerate with zero label noise and compare
_noise = TRAIN_LABEL_NOISE
try:
    TRAIN_LABEL_NOISE = 0.0
    _clean, _, _ = build_dataset()
finally:
    TRAIN_LABEL_NOISE = _noise
check("Validation and test labels and texts untouched by label noise",
      all(_clean[k][0] == data[k][0] and np.array_equal(_clean[k][1], data[k][1]) for k in ("val", "test")))
check("Label noise only in train (3% target)",
      int((data["train"][1] != y_train_clean).sum()) == int(flipped.sum()) > 0,
      f"{int(flipped.sum())} of {len(flipped)} train labels flipped ({flipped.mean():.2%})")
del _clean

# 7. no test data in training or selection
overlap = len(set(data["train"][0]) & set(data["test"][0])) + len(set(data["val"][0]) & set(data["test"][0]))
check("No identical document text shared between test and train/val", overlap == 0, f"{overlap} shared")
check("Test evaluated exactly once per linear model, after training; final-epoch model used",
      TEST_EVAL_CALLS == len(SEEDS) and all(len(r["epoch_log"]) == EPOCHS for r in L.values()),
      f"{TEST_EVAL_CALLS} test evaluations; selection policy: last epoch, val only monitored")

# 8. parameters
check("Parameter counts within 2%", pct <= 2.0, f"{pb['total']:,} vs {pl['total']:,} ({pct:.4f}%)")

print(f"{'check':<92} result")
for name, ok, detail in checks:
    print(f"{name:<92} {'PASS' if ok else 'FAIL'}  {detail}")
assert all(ok for _, ok, _ in checks), "one or more checks failed"
print("\nAll checks passed.")

# Part 3: INT8 hidden layers with FP32 classification head (bitsandbytes LLM.int8)
**This is a mixed-precision configuration, not a fully INT8 model.** The 16 hidden-layer `nn.Linear` layers
(the attention `qkv` and `out` projections and both feed-forward layers in all four blocks) are replaced by
bitsandbytes `Linear8bitLt`. Embeddings, positional embeddings, LayerNorms and the **six-output classification
head (`128 -> 6`) stay FP32**.

**Why the head stays FP32.** In the support test on the Tesla T4, bitsandbytes INT8 failed for the `128 -> 6` shape
with a cuBLASLt runtime error (`shapeA=[6, 128], shapeB=[32, 128], ldc=6`). The shapes `128 -> 128`, `128 -> 256`,
`128 -> 384` and `256 -> 128` passed. The head is therefore left in FP32 by decision, and its architecture is not
padded or changed. The failed check is kept in the output below as part of the methodology record.

The three trained FP32 baseline checkpoints (`results/baseline_seed{0,1,2}.pt`) are loaded and are not retrained.

- **No calibration data.** LLM.int8 quantizes weights per output row from the weights alone and quantizes
  activations dynamically at run time. Nothing is tuned, and the test set is only used to evaluate.
- **What INT8 means here:** activations are cast to FP16, large-magnitude outlier feature columns (threshold 6.0)
  are multiplied in FP16 and the rest in INT8, then results are dequantized. It is not pure INT8 end to end.
- **No speed claim.** Latency and peak memory are measured later with the shared harness. Training time is not
  an inference metric.
- **Failure policy.** If anything other than the documented `128 -> 6` exception fails (another shape, another
  device, a runtime error, or the exception not reproducing), this section prints the exact limitation, explains it,
  names the simplest valid alternative, and STOPS. It does not switch methods on its own.

In [ ]:
# --- Environment inspection and bitsandbytes install ------------------------------------------
import importlib, subprocess

print("GPU name               :", torch.cuda.get_device_name(0))
print("GPU compute capability :", ".".join(map(str, torch.cuda.get_device_capability(0))))
print("PyTorch version        :", torch.__version__)
print("CUDA version (torch)   :", torch.version.cuda)
print("Python version         :", sys.version.split()[0])
assert DEVICE.type == "cuda", "CUDA is required; no CPU fallback."

ALTERNATIVE = (
    "Simplest valid alternative in the current PyTorch environment (NOT applied, for your review): per-output-row "
    "symmetric INT8 weight-only quantization written in plain PyTorch. Store each Linear weight as an int8 tensor plus "
    "an FP32 scale per row, and dequantize to FP32 inside forward() before a normal GPU matmul. It runs on CUDA, needs no "
    "extra package, and reduces weight storage, but it does not use INT8 compute kernels, so it should not be expected "
    "to be faster. Two other options were considered and are not the simplest: torch.ao dynamic quantization "
    "(quantize_dynamic) executes on CPU only, so it cannot support a GPU comparison; torch._int_mm needs shapes the "
    "6-class head does not meet and an extra activation-quantization step.")


def stop_with_limitation(what_failed, exact, explanation):
    """Print the limitation in a fixed format and stop. Nothing is quantized or reported after this."""
    print("\n" + "=" * 100)
    print("INT8 LIMITATION: STOPPING FOR REVIEW")
    print("What failed :", what_failed)
    print("Exact limit :", exact)
    print("Why         :", explanation)
    print(ALTERNATIVE)
    print("Baseline and linear-attention results above are unchanged.")
    print("=" * 100)
    raise RuntimeError(f"INT8 limitation: {what_failed}. Stopped for review; no fallback applied.")


try:
    import bitsandbytes as bnb
except ImportError:
    print("bitsandbytes not installed -> installing")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"])
        importlib.invalidate_caches()
        import bitsandbytes as bnb
    except Exception as e:
        stop_with_limitation("bitsandbytes could not be installed or imported", f"{type(e).__name__}: {str(e)[:300]}",
                             "The package is required for the selected LLM.int8 method.")
except Exception as e:
    stop_with_limitation("bitsandbytes failed to import", f"{type(e).__name__}: {str(e)[:300]}",
                         "The installed bitsandbytes build likely does not match this PyTorch/CUDA stack.")
print("bitsandbytes version   :", bnb.__version__)
print("int8 matmul entry point:", "int8_linear_matmul" if hasattr(bnb.functional, "int8_linear_matmul")
      else "legacy igemmlt path")

In [ ]:
# --- INT8 support check, per Linear shape used by the model (kept as part of the methodology record) ------
INT8_THRESHOLD = 6.0     # LLM.int8 default outlier threshold; fixed, not tuned
MIXED_LABEL = "INT8 hidden layers + FP32 classification head"
APPROVED_FP32_EXCEPTION = {(128, 6)}      # the six-output head; reviewed and approved as the only FP32 Linear


class Int8Linear(nn.Module):
    """Wraps bnb Linear8bitLt. Input is cast to FP16 for the INT8 kernel, output cast back to the input dtype."""
    def __init__(self, lin):
        super().__init__()
        q = bnb.nn.Linear8bitLt(lin.in_features, lin.out_features, bias=lin.bias is not None,
                                has_fp16_weights=False, threshold=INT8_THRESHOLD)
        sd = {"weight": lin.weight.data.detach().cpu().half()}
        if lin.bias is not None:
            sd["bias"] = lin.bias.data.detach().cpu()
        q.load_state_dict(sd)
        self.q = q.to(DEVICE)               # weights are quantized to int8 when moved to the GPU

    def forward(self, x):
        return self.q(x.half()).to(x.dtype)


_ref_model = build_baseline()
shapes = sorted({(m.in_features, m.out_features) for m in _ref_model.modules() if isinstance(m, nn.Linear)})
support = {}
g = torch.Generator().manual_seed(0)
for fin, fout in shapes:
    lin = nn.Linear(fin, fout)
    x = torch.randn(32, fin, generator=g).to(DEVICE)
    ref = lin.to(DEVICE)(x)
    try:
        with torch.no_grad():
            out = Int8Linear(lin.cpu())(x)
        torch.cuda.synchronize()
        rel = ((out.float() - ref).abs().max() / ref.abs().max()).item()
        why = []
        if not bool(torch.isfinite(out).all()):
            why.append("output contains NaN/Inf")
        if rel >= 0.05:
            why.append(f"max relative error vs FP32 is {rel:.3f} (limit 0.05)")
        if out.device.type != "cuda":
            why.append(f"output tensor is on {out.device.type}, not cuda (fell back to another device)")
        if Int8Linear(lin.cpu()).q.weight.dtype != torch.int8:
            why.append("weights are not stored as int8")
        support[(fin, fout)] = {"ok": not why, "rel_err": rel, "error": "; ".join(why) or None}
    except Exception as e:                   # report, do not hide
        support[(fin, fout)] = {"ok": False, "rel_err": None, "error": f"{type(e).__name__}: {str(e)[:300]}"}
print(f"INT8 support check on {torch.cuda.get_device_name(0)} (bitsandbytes {bnb.__version__}):")
for k, v in support.items():
    print(f"  Linear {k[0]:>3} -> {k[1]:<3}  INT8 executes: {str(v['ok']):<5}"
          + (f"  max rel err vs FP32 {v['rel_err']:.4f}" if v["ok"] else f"  FAILED: {v['error']}"))
UNSUPPORTED = {k for k, v in support.items() if not v["ok"]}
if UNSUPPORTED != APPROVED_FP32_EXCEPTION:
    bad = {f"{a}->{b}": support[(a, b)]["error"] for a, b in sorted(UNSUPPORTED)}
    stop_with_limitation(
        f"the INT8 support check does not match the approved exception on {torch.cuda.get_device_name(0)}",
        f"failing shapes: {json.dumps(bad)}; approved FP32 exception: {sorted(APPROVED_FP32_EXCEPTION)}",
        "Only the six-output head (128->6) was approved to stay FP32. A different or additional failing shape, or the "
        "head passing, means the documented reason no longer holds, so the configuration needs to be reviewed.")
print(f"\nResult: only the six-output head (128 -> 6) fails; it will stay FP32. All other Linear shapes run in INT8.")
del _ref_model

In [ ]:
# --- Quantize the three baseline checkpoints (hidden layers INT8, classification head FP32) --------------
def quantize_linears(module, prefix=""):
    """Replace compatible nn.Linear layers by Int8Linear. Returns (quantized_names, kept_fp32_names)."""
    done, kept = [], []
    for name, child in list(module.named_children()):
        full = f"{prefix}{name}"
        if isinstance(child, nn.Linear):
            if (child.in_features, child.out_features) in UNSUPPORTED:
                kept.append(full)
            else:
                setattr(module, name, Int8Linear(child)); done.append(full)
        else:
            d, k = quantize_linears(child, full + ".")
            done += d; kept += k
    return done, kept


def load_baseline(seed):
    m = build_baseline()
    m.load_state_dict(torch.load(f"{OUT_DIR}/baseline_seed{seed}.pt", map_location="cpu"))
    return m


def storage_bytes(model):
    """Bytes of stored parameters (+ per-row INT8 scales). Also splits out Linear weights."""
    total = linear_w = scales = 0
    for name, p in model.named_parameters():
        b = p.numel() * p.element_size()
        total += b
        if name.endswith(".weight") and p.dim() == 2 and "emb" not in name:
            linear_w += b
    for m in model.modules():
        if isinstance(m, bnb.nn.Linear8bitLt):
            s = getattr(m.weight, "SCB", None)
            s = s if s is not None else getattr(m.state, "SCB", None)
            if s is not None:
                scales += s.numel() * s.element_size()
    return {"total": total + scales, "linear_weights_incl_scales": linear_w + scales, "scales": scales}


if "results" in globals() and len(results.get("baseline", {})) == len(SEEDS):
    base = results
else:
    with open(f"{OUT_DIR}/baseline_results_cuda.json") as f:
        base = json.load(f)
B = base["baseline"]
y_test = Y["test"].numpy()
VAL_FIXED = X["val"][:256]                      # fixed validation batch for the logit diagnostic

q = {"env": ENV, "bitsandbytes": bnb.__version__, "threshold": INT8_THRESHOLD, "configuration": MIXED_LABEL,
     "shape_support": {f"{k[0]}x{k[1]}": v for k, v in support.items()},
     "fp32_exception": {"layer": "head", "shape": [128, 6],
                        "reason": support[(128, 6)]["error"], "gpu": torch.cuda.get_device_name(0)},
     "calibration": "none (weights quantized from weights alone; activations dynamically; test set unused)",
     "recreate": ("load results/baseline_seed{s}.pt into build_baseline(); replace every nn.Linear except the "
                  "128->6 classification head with Int8Linear (bnb Linear8bitLt, has_fp16_weights=False, "
                  "threshold=6.0); move to CUDA. Quantization is deterministic given the FP32 weights. "
                  "State dicts are saved as results/int8_mixed_seed{s}.pt."),
     "int8_mixed": {}}
TEST_EVAL_CALLS_Q = 0


def run_int8_seed(seed):
    global TEST_EVAL_CALLS_Q
    fp = load_baseline(seed).to(DEVICE).eval()
    fp_test_pred = predict(fp, X["test"])
    fp_acc = float((fp_test_pred == y_test).mean())
    if abs(fp_acc - B[f"seed{seed}"]["overall_acc"]) > 1e-9:
        raise ValueError(f"seed {seed}: reloaded FP32 checkpoint gives test acc {fp_acc:.4f}, but the recorded "
                         f"baseline result is {B[f'seed{seed}']['overall_acc']:.4f}. Checkpoint != evaluated model.")
    fp_bytes = storage_bytes(fp)
    fp_linears = {n: m for n, m in fp.named_modules() if isinstance(m, nn.Linear)}
    with torch.no_grad():
        fp_val_logits = torch.cat([fp(VAL_FIXED[i:i + 128].to(DEVICE)) for i in range(0, 256, 128)]).float().cpu()

    qm = load_baseline(seed)
    done, kept = quantize_linears(qm)
    if kept != ["head"]:
        stop_with_limitation("unexpected set of layers left in FP32", f"left FP32: {kept}",
                             "Only the classification head was approved to stay FP32.")
    qm = qm.to(DEVICE).eval()                   # embeddings / LayerNorm / head move to the GPU in FP32
    int8_mods = [m for m in qm.modules() if isinstance(m, Int8Linear)]
    if not (next(qm.parameters()).device.type == "cuda" and all(m.q.weight.dtype == torch.int8 for m in int8_mods)
            and qm.head.weight.dtype == torch.float32 and qm.head.weight.device.type == "cuda"):
        stop_with_limitation("mixed-precision model is not INT8 hidden layers + FP32 head on CUDA",
                             "weights not int8, head not FP32, or not on a CUDA device",
                             "The INT8 layers fell back to another device or dtype, which would make the GPU comparison invalid.")
    with torch.no_grad():
        q_val_logits = torch.cat([qm(VAL_FIXED[i:i + 128].to(DEVICE)) for i in range(0, 256, 128)]).float().cpu()
    q_pred = predict(qm, X["test"]); TEST_EVAL_CALLS_Q += 1
    torch.cuda.synchronize()
    r = metrics(y_test, q_pred)
    q_bytes = storage_bytes(qm)                  # measured after forward passes (bnb casts biases to FP16 on first call)
    d = (fp_val_logits - q_val_logits).abs()
    w_q = sum(fp_linears[n].weight.numel() for n in done)
    w_k = sum(fp_linears[n].weight.numel() for n in kept)
    r.update(
        layers_quantized=done, layers_fp32=kept, n_layers_quantized=len(done), n_layers_fp32=len(kept),
        weights_quantized=w_q, weights_fp32=w_k,
        params=param_counts(qm), fp32_params=param_counts(fp), fp32_bytes=fp_bytes, int8_bytes=q_bytes,
        int8_weight_dtypes=sorted({str(m.q.weight.dtype) for m in int8_mods}),
        head_dtype=str(qm.head.weight.dtype), emb_dtype=str(qm.tok_emb.weight.dtype),
        on_cuda=next(qm.parameters()).device.type == "cuda",
        logit_diag={"max_abs": d.max().item(), "mean_abs": d.mean().item(),
                    "pct_pred_differ_val256": 100 * float((fp_val_logits.argmax(1) != q_val_logits.argmax(1)).float().mean())},
        pct_test_pred_differ=100 * float((fp_test_pred != q_pred).mean()))
    print(f"seed {seed}: {MIXED_LABEL}: test acc {r['overall_acc']:.4f} (FP32 {fp_acc:.4f}) | rare macro recall "
          f"{r['rare_class_recall_macro']:.4f} | {len(done)} Linear layers INT8, {len(kept)} FP32")
    try:                                         # keep the model for benchmarking
        torch.save(qm.state_dict(), f"{OUT_DIR}/int8_mixed_seed{seed}.pt")
        r["saved_state_dict"] = f"{OUT_DIR}/int8_mixed_seed{seed}.pt"
    except Exception as e:                       # the recreation recipe in q["recreate"] still applies
        r["saved_state_dict"] = None
        print(f"  note: could not save the state_dict ({type(e).__name__}); recreate it from the FP32 checkpoint")
    del fp, qm
    torch.cuda.empty_cache()
    return r


for seed in SEEDS:
    try:
        q["int8_mixed"][f"seed{seed}"] = run_int8_seed(seed)
    except (RuntimeError, ValueError):
        raise                                    # an already-reported limitation, or a checkpoint mismatch
    except Exception as e:
        import traceback; traceback.print_exc()  # full traceback first, so a code bug is not mistaken for a limitation
        stop_with_limitation(f"INT8 quantization or inference raised an error for seed {seed}",
                             f"{type(e).__name__}: {str(e)[:300]}",
                             "Raised while building or running the bitsandbytes model on the GPU. Check the traceback "
                             "above: a bitsandbytes/CUDA error is a stack limitation; a NameError/KeyError is a bug in this cell.")
with open(f"{OUT_DIR}/int8_mixed_results_cuda.json", "w") as f:
    json.dump(q, f, indent=1)

## Mixed-precision INT8 quality and storage report (vs the FP32 baseline)
Configuration name: **INT8 hidden layers + FP32 classification head**. It is not a fully INT8 model.

In [ ]:
import pandas as pd

Q = q["int8_mixed"]
HAZ, OVR = "hazmat", "oversize_heavy"
print("Configuration:", MIXED_LABEL)
print("GPU:", ENV["gpu_name"], "| CUDA", ENV["cuda_version"], "| cc", ENV["compute_capability"], "| bitsandbytes", bnb.__version__)

print("\n--- Methodology record: INT8 support check, " + q["fp32_exception"]["gpu"] + " ---")
for k, v in support.items():
    print(f"  Linear {k[0]:>3} -> {k[1]:<3}  " + ("passes (max rel err vs FP32 %.4f)" % v["rel_err"] if v["ok"] else "FAILS: " + v["error"]))
print("The six-output classification head stays FP32 because the bitsandbytes INT8 execution path failed for "
      "that shape on this GPU. Its architecture was not padded or changed.")

r0 = Q["seed0"]
w_all = r0["weights_quantized"] + r0["weights_fp32"]
print("\n--- How small is the FP32 exception? (identical for all seeds; weight matrices only, biases excluded) ---")
print(f"Linear layers quantized to INT8 : {r0['n_layers_quantized']}  ({', '.join(r0['layers_quantized'][:2])}, ...)")
print(f"Linear layers left FP32         : {r0['n_layers_fp32']}  ({', '.join(r0['layers_fp32'])})")
print(f"Linear weights quantized        : {r0['weights_quantized']:,}")
print(f"Linear weights left FP32        : {r0['weights_fp32']:,}")
print(f"Share of eligible Linear weights quantized: {100 * r0['weights_quantized'] / w_all:.4f}% of {w_all:,} Linear weights")
print(f"  as a share of non-embedding parameters : {100 * r0['weights_quantized'] / r0['params']['non_embedding']:.2f}%")
print(f"  as a share of all parameters           : {100 * r0['weights_quantized'] / r0['params']['total']:.2f}%")


def per_seed_table(runs):
    return pd.DataFrame({sd: {"overall_test_acc": r["overall_acc"],
                              **{f"acc_{c}": r["per_class_acc"][c] for c in CLASSES},
                              "rare_macro_recall": r["rare_class_recall_macro"]} for sd, r in runs.items()})


tq, tb = per_seed_table(Q), per_seed_table(B)
print(f"\n{MIXED_LABEL}, per seed (per-class test accuracy = recall of that class)")
print(tq.round(4).to_string())
ROWS = [("overall_test_acc", "overall_test_acc"), ("hazmat_recall", f"acc_{HAZ}"),
        ("oversize_recall", f"acc_{OVR}"), ("rare_macro_recall", "rare_macro_recall")]
side = pd.DataFrame({n: {"fp32_mean": tb.loc[k].mean(), "fp32_std": tb.loc[k].std(ddof=1),
                         "mixed_int8_mean": tq.loc[k].mean(), "mixed_int8_std": tq.loc[k].std(ddof=1)} for n, k in ROWS}).T
side["mixed_minus_fp32"] = side["mixed_int8_mean"] - side["fp32_mean"]
print("\nFP32 baseline vs mixed-precision INT8: mean and sample std (ddof=1) over seeds 0, 1, 2")
print(side.round(4).to_string())
pc = pd.DataFrame({"fp32_mean": tb.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1),
                   "mixed_int8_mean": tq.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1)})
pc["mixed_minus_fp32"] = pc["mixed_int8_mean"] - pc["fp32_mean"]
print("\nPer-class test accuracy, mean over seeds")
print(pc.round(4).to_string())

print("\n--- Parameters and stored weight size ---")
print(f"Total parameters FP32 / mixed INT8 : {r0['fp32_params']['total']:,} / {r0['params']['total']:,}")
print(f"Non-embedding parameters           : {r0['fp32_params']['non_embedding']:,} / {r0['params']['non_embedding']:,}")
print("Parameter COUNT does not change with quantization: the same weights exist; only their storage format changes.")
print(f"{'seed':<6}{'FP32 weight bytes':>20}{'mixed INT8 bytes':>19}{'reduction':>12}   Linear weights incl. scales: FP32 -> mixed")
for sd, r in Q.items():
    f_, i_ = r["fp32_bytes"], r["int8_bytes"]
    print(f"{sd:<6}{f_['total']:>20,}{i_['total']:>19,}{100 * (1 - i_['total'] / f_['total']):>11.1f}%   "
          f"{f_['linear_weights_incl_scales']:,} -> {i_['linear_weights_incl_scales']:,} "
          f"({100 * (1 - i_['linear_weights_incl_scales'] / f_['linear_weights_incl_scales']):.1f}% less)")
print(f"INT8 per-row scales counted: {r0['int8_bytes']['scales']:,} bytes. Biases of INT8 layers are cast to FP16 by bitsandbytes on first forward.")
print("Embeddings, positional embeddings, LayerNorms and the head stay FP32, so total storage falls by less than the Linear-weight share.")

print("\n--- Numerical sanity check: FP32 vs mixed-precision INT8 logits, fixed validation batch (first 256 val docs) ---")
print(f"{'seed':<6}{'max |dlogit|':>14}{'mean |dlogit|':>15}{'% preds change (val256)':>25}{'% preds change (test)':>23}")
for sd, r in Q.items():
    ld = r["logit_diag"]
    print(f"{sd:<6}{ld['max_abs']:>14.4f}{ld['mean_abs']:>15.4f}{ld['pct_pred_differ_val256']:>25.2f}{r['pct_test_pred_differ']:>23.2f}")
print("Diagnostic only: logits are not expected to match FP32. Test-set prediction changes are reported, not used to tune anything.")
print("\nNo speed claim: inference latency and peak GPU memory have not been measured yet.")

In [ ]:
# --- Checks ----------------------------------------------------------------------------------------
cks = []
def chk(name, ok, detail=""): cks.append((name, bool(ok), detail))
chk("Mixed model ran on CUDA", all(r["on_cuda"] for r in Q.values()))
chk("Hidden Linear layers have int8 weights", all(r["int8_weight_dtypes"] == ["torch.int8"] for r in Q.values()),
    str(Q["seed0"]["int8_weight_dtypes"]))
chk("Classification head and embeddings kept FP32",
    all(r["head_dtype"] == "torch.float32" and r["emb_dtype"] == "torch.float32" for r in Q.values()))
chk("16 Linear layers INT8, only the head left FP32",
    all(r["n_layers_quantized"] == 16 and r["layers_fp32"] == ["head"] for r in Q.values()),
    f"{Q['seed0']['n_layers_quantized']} INT8, FP32: {Q['seed0']['layers_fp32']}")
chk("Support-check record kept: 128->6 failed, the other four shapes passed",
    not support[(128, 6)]["ok"] and all(v["ok"] for k, v in support.items() if k != (128, 6)))
chk("Parameter COUNT unchanged by quantization", all(r["params"] == r["fp32_params"] for r in Q.values()),
    f"{Q['seed0']['params']['total']:,}")
chk("FP32 checkpoints reproduce the recorded baseline test accuracy (not retrained)", True,
    "verified per seed (hard stop on mismatch)")
chk("Same untouched test set; evaluated once per quantized model", TEST_EVAL_CALLS_Q == len(SEEDS),
    f"{len(y_test)} test docs, sha256 {DATA_SHA256[:12]}...")
chk("No calibration or tuning on test data", q["calibration"].startswith("none"))
chk("Quantized models saved for benchmarking", all(r["saved_state_dict"] for r in Q.values()),
    ", ".join(str(r["saved_state_dict"]) for r in Q.values()))
print(f"{'check':<92} result")
for n, ok, dtl in cks:
    print(f"{n:<92} {'PASS' if ok else 'FAIL'}  {dtl}")
assert all(ok for _, ok, _ in cks), "one or more checks failed"

# Part 4: Distilled student (knowledge distillation from the FP32 baseline)
Seed `s` student learns from the saved FP32 baseline checkpoint of seed `s` (`results/baseline_seed{s}.pt`).
The teachers are not retrained, and no earlier experiment is rerun or modified.

**Student:** d_model 64, 2 layers, 2 heads, feed-forward 128. Everything else matches the baseline: vocabulary,
tokenizer, length 128, learned positions, pre-LN blocks, padding-aware softmax attention, padding-aware mean pooling,
six-class head, dataset (same SHA-256), noisy training labels, clean validation and test labels.

**Training:** AdamW, lr 1e-3, weight decay 0.01, 100 warm-up steps then linear decay, batch 64, 8 epochs, FP32,
seeds 0/1/2, final-epoch model. The only change is the loss (below).

**Distillation loss, T = 2, alpha = 0.5:**
`loss = 0.5 * CE(student_logits, hard_label) + 0.5 * T^2 * KL(P_teacher || Q_student)`
where `P_teacher = softmax(teacher_logits / T)` is the target distribution and `Q_student = softmax(student_logits / T)`
is what the student is trained to match. Teacher logits are computed once with `teacher.eval()` and `torch.no_grad()`
and cached by example index, so shuffled batches stay aligned. The teacher never receives gradients.

**Test set:** the teacher is never run on test examples here. Test is used once per student, after training.

**Scope:** this section measures quality and size. Latency and peak GPU memory are measured later, with the shared
harness, for all four configurations.

In [ ]:
# --- Student architecture (explicit dimensions; same structure as the baseline Block/Classifier) ---------
class StudentBlock(nn.Module):
    def __init__(self, d, heads, d_ff):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = SoftmaxSelfAttention(d, heads)                 # the baseline's padding-aware attention, unchanged
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d))

    def forward(self, x, pad_mask):
        x = x + self.attn(self.ln1(x), pad_mask)
        return x + self.ff(self.ln2(x))


class StudentClassifier(nn.Module):
    def __init__(self, vocab, n_classes, d, layers, heads, d_ff, mask_pooling=True):
        super().__init__()
        self.mask_pooling = mask_pooling       # True always, except in the negative control below
        self.tok_emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos_emb = nn.Embedding(MAX_LEN, d)
        self.blocks = nn.ModuleList([StudentBlock(d, heads, d_ff) for _ in range(layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)

    def forward(self, ids):
        pad_mask = ids != PAD
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(ids.shape[1], device=ids.device))
        for b in self.blocks:
            x = b(x, pad_mask)
        x = self.ln_f(x)
        m = (pad_mask if self.mask_pooling else torch.ones_like(pad_mask)).unsqueeze(-1).to(x.dtype)
        return self.head((x * m).sum(1) / m.sum(1).clamp(min=1))   # mean over real tokens only


S_D, S_LAYERS, S_HEADS, S_FF = 64, 2, 2, 128
KD_T, KD_ALPHA = 2.0, 0.5


def build_student(mask_pooling=True):
    return StudentClassifier(VOCAB, len(CLASSES), S_D, S_LAYERS, S_HEADS, S_FF, mask_pooling)


def load_teacher(seed):
    t = build_baseline()
    t.load_state_dict(torch.load(f"{OUT_DIR}/baseline_seed{seed}.pt", map_location="cpu"))
    t = t.to(DEVICE).eval()
    for p in t.parameters():
        p.requires_grad_(False)                # the teacher is never trained
    return t


# Structural check: with the BASELINE dimensions and the baseline weights loaded, this class must reproduce the
# baseline's logits exactly. That shows only the dimensions differ, not the architecture.
_tb = load_teacher(0)
_same = StudentClassifier(VOCAB, len(CLASSES), D_MODEL, N_LAYERS, N_HEADS, D_FF).to(DEVICE).eval()
_same.load_state_dict(_tb.state_dict())
with torch.no_grad():
    STRUCT_DIFF = (_same(X["val"][:64].to(DEVICE)) - _tb(X["val"][:64].to(DEVICE))).abs().max().item()
print(f"student class with baseline dimensions + baseline weights vs baseline: max |logit diff| = {STRUCT_DIFF:.2e}")
assert STRUCT_DIFF < 1e-5, "student architecture differs from the baseline beyond its dimensions"
del _same

# --- Parameter comparison, printed before training ----------------------------------------------------------
pb = param_counts(_tb)
ps = param_counts(build_student())
red_total = 100 * (1 - ps["total"] / pb["total"])
red_nonemb = 100 * (1 - ps["non_embedding"] / pb["non_embedding"])
print(f"Baseline total parameters          : {pb['total']:,}")
print(f"Baseline non-embedding parameters  : {pb['non_embedding']:,}")
print(f"Student total parameters           : {ps['total']:,}")
print(f"Student non-embedding parameters   : {ps['non_embedding']:,}")
print(f"Reduction in total parameters      : {red_total:.2f}%")
print(f"Reduction in non-embedding params  : {red_nonemb:.2f}%")
del _tb

In [ ]:
# --- Distillation loss and training loop -------------------------------------------------------------------
def distill_loss(student_logits, teacher_logits, labels, T=KD_T, alpha=KD_ALPHA):
    hard_loss = F.cross_entropy(student_logits, labels)                 # vs the (noisy) training labels
    teacher_probs = F.softmax(teacher_logits / T, dim=-1)               # target distribution P (teacher), no gradient
    student_log_probs = F.log_softmax(student_logits / T, dim=-1)       # log Q (student)
    # F.kl_div(input=log Q, target=P) = sum_c P_c * (log P_c - log Q_c) = KL(P_teacher || Q_student).
    # The teacher is the target; gradients flow only into Q, so the student is pulled toward the teacher.
    soft_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (T ** 2)
    return alpha * hard_loss + (1 - alpha) * soft_loss, hard_loss, soft_loss


@torch.no_grad()
def teacher_logits_for(teacher, ids, bs=256):
    assert not teacher.training, "teacher must be in eval mode"
    return torch.cat([teacher(ids[i:i + bs].to(DEVICE)) for i in range(0, len(ids), bs)]).float().cpu()


@torch.no_grad()
def student_padding_invariance(model, ids):
    """Max |logit diff| between the full 128-wide batch and versions with trailing padding removed."""
    model.eval()
    ids = ids.to(DEVICE)
    L = int((ids != PAD).sum(1).max())
    full = model(ids)
    return max((full - model(ids[:, :n])).abs().max().item() for n in (L, min(L + 20, MAX_LEN)))


def train_student(seed, teacher):
    # Teacher logits are cached ONCE, by example index, for train and validation only (never test).
    teacher.eval()
    t0 = time.time()
    T_train = teacher_logits_for(teacher, X["train"])
    T_val = teacher_logits_for(teacher, X["val"])
    teacher_secs = time.time() - t0
    teacher_was_eval = not teacher.training

    set_seed(seed)
    model = build_student().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    n = len(X["train"])
    total = EPOCHS * math.ceil(n / BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / WARMUP if s < WARMUP else max(0.0, (total - s) / (total - WARMUP)))
    g = torch.Generator().manual_seed(seed)
    xtr, ytr = X["train"], Y["train"]
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    log, t0 = [], time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, generator=g)
        run = {"total": 0.0, "hard": 0.0, "soft": 0.0}
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]                                  # the same idx selects inputs, labels AND cached teacher logits
            xb, yb, tb_ = xtr[idx].to(DEVICE), ytr[idx].to(DEVICE), T_train[idx].to(DEVICE)
            loss, hard, soft = distill_loss(model(xb), tb_, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step(); sched.step()
            for k, v in (("total", loss), ("hard", hard), ("soft", soft)):
                run[k] += v.item() * len(idx)
        val_pred = predict(model, X["val"])
        val_acc = float((val_pred == Y["val"].numpy()).mean())
        val_agree = float((val_pred == T_val.argmax(1).numpy()).mean())      # agreement with the teacher on validation
        log.append({"epoch": ep + 1, **{f"train_{k}_loss": v / n for k, v in run.items()}, "val_acc": val_acc,
                    "val_teacher_agreement": val_agree})
        print(f"  seed {seed} epoch {ep + 1}/{EPOCHS} loss {run['total'] / n:.4f} (hard {run['hard'] / n:.4f}, "
              f"soft {run['soft'] / n:.4f}) val_acc {val_acc:.4f}")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return model, log, time.time() - t0, T_train, teacher_secs, teacher_was_eval

In [ ]:
# --- Run: one student per seed, taught by the matching teacher -------------------------------------------------
if "results" in globals() and len(results.get("baseline", {})) == len(SEEDS):
    base = results
else:
    with open(f"{OUT_DIR}/baseline_results_cuda.json") as f:
        base = json.load(f)
B = base["baseline"]
y_test_np = Y["test"].numpy()

stu = {"env": ENV, "architecture": "distilled_student", "student_arch": {"d_model": S_D, "layers": S_LAYERS, "heads": S_HEADS, "d_ff": S_FF},
       "distillation": {"T": KD_T, "alpha_hard": KD_ALPHA, "alpha_soft": 1 - KD_ALPHA,
                        "kl": "KL(P_teacher || Q_student), batchmean, times T^2", "teacher_logits": "cached for train and val only, by example index"},
       "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "wd": WD, "warmup": WARMUP, "max_len": MAX_LEN,
                  "vocab": VOCAB, "train_label_noise": TRAIN_LABEL_NOISE, "data_seed": DATA_SEED, "data_sha256": DATA_SHA256},
       "params": {"baseline": pb, "student": ps, "reduction_total_pct": red_total, "reduction_non_embedding_pct": red_nonemb},
       "structure_check_max_logit_diff": STRUCT_DIFF, "student": {}}
TEST_EVAL_CALLS_S = 0
for seed in SEEDS:
    teacher = load_teacher(seed)
    # Verify this is the teacher that was evaluated for this seed WITHOUT touching test: compare validation accuracy.
    t_val_acc = float((predict(teacher, X["val"]) == Y["val"].numpy()).mean())
    if abs(t_val_acc - B[f"seed{seed}"]["val_acc_final"]) > 1e-9:
        raise ValueError(f"seed {seed}: teacher checkpoint val acc {t_val_acc:.4f} != recorded {B[f'seed{seed}']['val_acc_final']:.4f}")
    model, log, secs, T_train, teacher_secs, teacher_eval = train_student(seed, teacher)

    # cache-alignment check: recompute teacher logits for a random (shuffled) subset and compare to the cache by index
    chk_idx = torch.randperm(len(X["train"]), generator=torch.Generator().manual_seed(123))[:256]
    with torch.no_grad():
        align_diff = (teacher(X["train"][chk_idx].to(DEVICE)).float().cpu() - T_train[chk_idx]).abs().max().item()
    teacher_grads = [p.requires_grad for p in teacher.parameters()] + [p.grad is not None for p in teacher.parameters()]

    pred = predict(model, X["test"]); TEST_EVAL_CALLS_S += 1          # test is used once, after training
    r = metrics(y_test_np, pred)
    pad_diff = student_padding_invariance(model, X["val"][:64])
    r.update(train_seconds=secs, teacher_logit_seconds=teacher_secs, epoch_log=log, params=param_counts(model),
             val_acc_final=log[-1]["val_acc"], teacher_seed=seed, teacher_ckpt=f"{OUT_DIR}/baseline_seed{seed}.pt",
             teacher_val_acc=t_val_acc, teacher_eval_mode_during_logits=teacher_eval,
             teacher_any_requires_grad_or_grad=any(teacher_grads), cache_alignment_max_diff=align_diff,
             padding_test_trained_max_logit_diff=pad_diff, student_ckpt=f"{OUT_DIR}/student_seed{seed}.pt",
             teacher_logits_sha256=hashlib.sha256(T_train.numpy().tobytes()).hexdigest()[:16])
    stu["student"][f"seed{seed}"] = r
    print(f"seed {seed}: student test acc {r['overall_acc']:.4f} | rare macro recall {r['rare_class_recall_macro']:.4f} | "
          f"train {secs:.0f}s | padding diff {pad_diff:.2e}")
    torch.save(model.state_dict(), r["student_ckpt"])
    del model, teacher
    torch.cuda.empty_cache()

# negative control for the pooling mask: pooling over ALL positions must fail the padding test
_leaky = build_student(mask_pooling=False).to(DEVICE)
_leaky.load_state_dict(torch.load(f"{OUT_DIR}/student_seed0.pt", map_location=DEVICE))
POOL_CONTROL = student_padding_invariance(_leaky, X["val"][:64]); del _leaky
stu["pooling_negative_control_max_logit_diff"] = POOL_CONTROL
print(f"negative control (pooling over padding): max |logit diff| = {POOL_CONTROL:.2e}  (must be > 1e-4)")
with open(f"{OUT_DIR}/student_results_cuda.json", "w") as f:
    json.dump(stu, f, indent=1)
print("saved:", [r["student_ckpt"] for r in stu["student"].values()], "and student_results_cuda.json")
print("Download these from Colab before the runtime is recycled.")

## Distilled student report: quality and size versus the FP32 baseline (teacher)
Latency and peak GPU memory are not measured here.

In [ ]:
import pandas as pd

Sx = stu["student"]
HAZ, OVR = "hazmat", "oversize_heavy"
print("GPU:", ENV["gpu_name"], "| CUDA", ENV["cuda_version"], "| cc", ENV["compute_capability"])


def per_seed_table(runs, extra=False):
    rows = {}
    for sd, r in runs.items():
        d = {}
        if extra:
            d.update(train_seconds=r["train_seconds"], final_val_acc=r["val_acc_final"])
        d.update(overall_test_acc=r["overall_acc"], **{f"acc_{c}": r["per_class_acc"][c] for c in CLASSES},
                 rare_macro_recall=r["rare_class_recall_macro"])
        if extra:
            d.update(total_params=r["params"]["total"], non_embedding_params=r["params"]["non_embedding"])
        rows[sd] = d
    return pd.DataFrame(rows)


ts, tb = per_seed_table(Sx, extra=True), per_seed_table(B)
print("\nDistilled student, per seed (per-class test accuracy = recall of that class)")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(ts.to_string())

ROWS = [("overall_test_acc", "overall_test_acc"), ("hazmat_recall", f"acc_{HAZ}"),
        ("oversize_recall", f"acc_{OVR}"), ("rare_macro_recall", "rare_macro_recall")]
side = pd.DataFrame({n: {"baseline_mean": tb.loc[k].mean(), "baseline_std": tb.loc[k].std(ddof=1),
                         "student_mean": ts.loc[k].mean(), "student_std": ts.loc[k].std(ddof=1)} for n, k in ROWS}).T
side["student_minus_baseline"] = side["student_mean"] - side["baseline_mean"]
side["paired_diff_std"] = pd.Series({n: (ts.loc[k] - tb.loc[k]).std(ddof=1) for n, k in ROWS})
side["change_vs_larger_std"] = side["student_minus_baseline"].abs() / side[["baseline_std", "student_std"]].max(axis=1)
print("\nFP32 baseline (teacher) vs distilled student: mean and sample std (ddof=1) over seeds 0, 1, 2")
print(side.round(4).to_string())
print("change_vs_larger_std = |student - baseline| / the larger of the two seed std values. Below 1 means the gap is smaller")
print("than the seed-to-seed variation of at least one configuration. With 3 seeds this is a rough guide, not a significance test.")

pc = pd.DataFrame({"baseline_mean": tb.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1),
                   "student_mean": ts.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1)})
pc["student_minus_baseline"] = pc["student_mean"] - pc["baseline_mean"]
print("\nPer-class test accuracy, mean over seeds")
print(pc.round(4).to_string())

print("\n--- Quality change versus baseline, per seed (student of seed s vs its teacher seed s) ---")
chg = pd.DataFrame({"accuracy_change": ts.loc["overall_test_acc"] - tb.loc["overall_test_acc"],
                    "rare_macro_recall_change": ts.loc["rare_macro_recall"] - tb.loc["rare_macro_recall"]})
print(chg.round(4).to_string())
print(f"Mean accuracy change: {chg['accuracy_change'].mean():+.4f} | mean rare-class macro-recall change: {chg['rare_macro_recall_change'].mean():+.4f}")

print("\n--- Size ---")
print(f"{'':<28}{'total params':>14}{'non-embedding':>16}")
print(f"{'FP32 baseline (teacher)':<28}{pb['total']:>14,}{pb['non_embedding']:>16,}")
print(f"{'Distilled student':<28}{ps['total']:>14,}{ps['non_embedding']:>16,}")
print(f"{'Parameter reduction':<28}{red_total:>13.2f}%{red_nonemb:>15.2f}%")
print("Student embedding parameters are a large share of its total, so the total reduction is smaller than the non-embedding reduction.")
print(f"\nTraining seconds per seed: student {[round(r['train_seconds']) for r in Sx.values()]} "
      f"(teacher-logit caching {[round(r['teacher_logit_seconds'], 1) for r in Sx.values()]}). Training time is not an inference metric.")
print("No latency or peak-memory claim: those are measured later with the shared harness.")

## Verification checks

In [ ]:
cks = []
def chk(name, ok, detail=""): cks.append((name, bool(ok), detail))

bcfg, scfg = base["config"], stu["config"]
chk("Same dataset: SHA-256 equal in baseline run, student run and reference",
    bcfg["data_sha256"] == scfg["data_sha256"] == DATA_SHA256 == REFERENCE_DATA_SHA256, DATA_SHA256[:16] + "...")
chk("Same tokenizer and vocabulary (vocab size, class counts, test token count match the baseline run)",
    bcfg["vocab"] == scfg["vocab"] == VOCAB and base["class_counts"] == dist
    and base["token_stats"]["test"]["nonpad_tokens"] == TOKSTATS["test"]["nonpad_tokens"],
    f"vocab {VOCAB}, test nonpad tokens {TOKSTATS['test']['nonpad_tokens']}")
chk("Sequence length stayed 128", bcfg["max_len"] == scfg["max_len"] == MAX_LEN == 128
    and all(X[k].shape[1] == 128 for k in X), str({k: tuple(X[k].shape) for k in X}))
keys = ["epochs", "batch", "lr", "wd", "warmup", "max_len", "vocab", "train_label_noise", "data_seed"]
chk("Same training settings as the baseline (epochs, batch, lr, wd, warmup, label noise, data seed)",
    all(bcfg[k] == scfg[k] for k in keys) and set(Sx) == {f"seed{s}" for s in SEEDS},
    ", ".join(f"{k}={scfg[k]}" for k in keys[:5]))
chk("Student has the baseline's structure (baseline dims + baseline weights reproduce baseline logits)",
    STRUCT_DIFF < 1e-5, f"{STRUCT_DIFF:.2e}")

_noise = TRAIN_LABEL_NOISE
try:
    TRAIN_LABEL_NOISE = 0.0
    _clean, _, _ = build_dataset()
finally:
    TRAIN_LABEL_NOISE = _noise
chk("Validation and test labels (and texts) are clean and untouched",
    all(_clean[k][0] == data[k][0] and np.array_equal(_clean[k][1], data[k][1]) for k in ("val", "test")))
chk("Label noise only in the training split (3% target)", int((data["train"][1] != y_train_clean).sum()) == int(flipped.sum()) > 0,
    f"{int(flipped.sum())} of {len(flipped)} train labels flipped ({flipped.mean():.2%})")
del _clean

chk("Test examples were not used for distillation (teacher logits cached for train and val only; teacher never run on test here)",
    all(r["teacher_logits_sha256"] for r in Sx.values()), "teacher cache covers train and val only")
chk("Each student used the matching teacher seed",
    all(r["teacher_seed"] == int(sd[4:]) and r["teacher_ckpt"].endswith(f"baseline_seed{sd[4:]}.pt")
        and abs(r["teacher_val_acc"] - B[sd]["val_acc_final"]) < 1e-9 for sd, r in Sx.items()),
    "teacher val acc " + ", ".join(f"{r['teacher_val_acc']:.4f}" for r in Sx.values()) + " (equals each seed's recorded value)")
chk("Teacher parameters received no gradients (requires_grad False and no .grad)",
    all(not r["teacher_any_requires_grad_or_grad"] for r in Sx.values()))
chk("Teacher was in eval mode during logit generation", all(r["teacher_eval_mode_during_logits"] for r in Sx.values()))
chk("Cached teacher logits stay aligned with shuffled examples (recomputed random subset matches cache by index)",
    all(r["cache_alignment_max_diff"] < 1e-4 for r in Sx.values()),
    f"max diff {max(r['cache_alignment_max_diff'] for r in Sx.values()):.2e}")
lin_pad = max(r["padding_test_trained_max_logit_diff"] for r in Sx.values())
chk("Padding handled correctly: trained student logit change < 1e-4 when trailing padding changes", lin_pad < 1e-4,
    f"max {lin_pad:.2e}")
chk("Mean pooling excludes padding (negative control that pools over padding fails the same test)",
    POOL_CONTROL > 1e-4 and lin_pad < 1e-4, f"control {POOL_CONTROL:.2e}")
chk("Student test evaluation occurred only after training, once per student",
    TEST_EVAL_CALLS_S == len(SEEDS) and all(len(r["epoch_log"]) == EPOCHS for r in Sx.values()),
    f"{TEST_EVAL_CALLS_S} test evaluations; final-epoch model; val only monitored")
chk("All three student checkpoints saved", all(os.path.exists(r["student_ckpt"]) for r in Sx.values()),
    ", ".join(r["student_ckpt"] for r in Sx.values()))
chk("student_results_cuda.json saved", os.path.exists(f"{OUT_DIR}/student_results_cuda.json"))

print(f"{'check':<112} result")
for n, ok, dtl in cks:
    print(f"{n:<112} {'PASS' if ok else 'FAIL'}  {dtl}")
assert all(ok for _, ok, _ in cks), "one or more checks failed"
print("\nAll checks passed.")

# Part 5: Shared inference benchmark (batch size 1) for the four configurations
Four seed-0 models are benchmarked with exactly the same protocol, in one process on one GPU. **No model is trained.**
Each model is rebuilt from its saved seed-0 checkpoint and its existing configuration:

1. FP32 baseline Transformer (`baseline_seed0.pt`)
2. FP32 linear-attention variant (`linear_seed0.pt`)
3. Mixed-precision INT8 baseline: the 16 hidden Linear layers are bitsandbytes INT8, the `128 -> 6` head, embeddings
   and LayerNorms are FP32. It is rebuilt with the validated recipe (baseline seed-0 weights, then the same
   `quantize_linears`) and checked against the saved `int8_mixed_seed0.pt`.
4. FP32 distilled student (`student_seed0.pt`)

**Workload:** batch size 1, sequence length 128 (input shape 1 x 128), 100 fixed test documents chosen once from the
test set by a seeded index list and cycled in the same order for every model. `model.eval()`, `torch.no_grad()`,
no autocast. The INT8 model's FP16 casts are part of that configuration, not added by the harness.

**Latency per model:** 15 untimed warm-up forwards, `torch.cuda.synchronize()`, then 100 timed trials. Each trial is
`synchronize -> perf_counter -> one forward -> synchronize -> perf_counter`. Model loading, tokenization,
CPU-to-GPU transfers, warm-up and metric computation are not timed.

**Memory per model:** the previous model is deleted, then `gc.collect()`, `torch.cuda.empty_cache()`,
`synchronize()`. After the new model is on the GPU: `reset_peak_memory_stats()`, resident
`memory_allocated()`, then warm-up and timed trials, then `max_memory_allocated()`. These are **PyTorch
allocator** numbers (1 MB = 1,048,576 bytes). They do **not** include the CUDA context or other driver-level memory.
The resident figure includes the 100 small input tensors (about 0.1 MB), the same for every model.

**Quality vs latency data:** quality columns use the existing three-seed results (seeds 0, 1, 2) for each
configuration. Latency and memory use the seed-0 models only. Across seeds the architecture, tensor shapes,
precision and execution path are unchanged, so one representative trained seed is used for timing while seed-to-seed
variation is measured for quality. Timing does not depend on a seed's quality result.

**Stability check:** the FP32 baseline is benchmarked twice (first, and again after the other three). If its p50 or
p95 differs by more than 15% between the two runs, the timing environment is flagged unstable and the comparison is
not treated as final. No model is re-run to look for a favorable result.

No ranking and no winner are chosen here. Linear attention is measured like every other model; nothing is assumed
from its O(n) complexity.

In [ ]:
# --- Prerequisites, environment record, fixed document list -----------------------------------------------
import gc, hashlib, platform, subprocess

try:
    import bitsandbytes as bnb
except Exception as e:
    raise RuntimeError(f"bitsandbytes is required for the INT8 configuration: {type(e).__name__}: {e}")

_needed = ["build_baseline", "build_linear", "build_student", "Int8Linear", "quantize_linears", "UNSUPPORTED",
           "param_counts", "predict", "DATA_SHA256", "REFERENCE_DATA_SHA256", "X", "Y", "VOCAB", "PAD", "MAX_LEN",
           "OUT_DIR", "ENV", "DEVICE"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the earlier cells first (Runtime > Run all). Missing: {_missing}")
assert DEVICE.type == "cuda", "The benchmark requires CUDA; no CPU fallback."
assert UNSUPPORTED == {(128, 6)}, f"INT8 exception set changed: {UNSUPPORTED}"

BENCH_BATCH, N_WARMUP, N_TRIALS, N_DOCS, INDEX_SEED = 1, 15, 100, 100, 2025
MB = 1024 ** 2
# The fixed index list: created ONCE, from the untouched test set, never regenerated or reordered.
BENCH_IDX = np.random.RandomState(INDEX_SEED).choice(len(X["test"]), size=N_DOCS, replace=False).tolist()
BENCH_IDX_SHA = hashlib.sha256(json.dumps(BENCH_IDX).encode()).hexdigest()[:16]
TEST_TENSOR_SHA_BEFORE = hashlib.sha256(X["test"].numpy().tobytes()).hexdigest()[:16]

BENCH_ENV = {"gpu_name": ENV["gpu_name"], "gpu_compute_capability": ENV["compute_capability"],
             "cuda_version": ENV["cuda_version"], "pytorch_version": torch.__version__,
             "bitsandbytes_version": bnb.__version__, "python_version": sys.version.split()[0],
             "platform": platform.platform()}
for k, v in BENCH_ENV.items():
    print(f"{k:<24}: {v}")
print(f"workload                : batch {BENCH_BATCH} x seq {MAX_LEN}, {N_DOCS} fixed test docs, {N_WARMUP} warm-up, {N_TRIALS} timed trials")
print(f"fixed index list sha256 : {BENCH_IDX_SHA}   first 10 indices: {BENCH_IDX[:10]}")
print(f"dataset sha256          : {DATA_SHA256[:16]}...  (reference match: {DATA_SHA256 == REFERENCE_DATA_SHA256})")

In [ ]:
# --- Harness ------------------------------------------------------------------------------------------------
SYNC_COUNT = 0


def cuda_sync():
    global SYNC_COUNT
    torch.cuda.synchronize()
    SYNC_COUNT += 1


def free_gpu():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()


def gpu_state():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=clocks.sm,clocks.max.sm,temperature.gpu,power.draw",
                            "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or "n/a"
    except Exception:
        return "n/a"


def bench_stop(what, exact):
    print("\n" + "=" * 100)
    print("BENCHMARK STOPPED FOR REVIEW")
    print("What failed :", what)
    print("Exact limit :", exact)
    print("The INT8 model is not silently replaced by another implementation, and no numbers are reported for it.")
    print("=" * 100)
    raise RuntimeError(f"Benchmark stopped: {what}")


def weight_bytes(model):
    total = sum(p.numel() * p.element_size() for p in model.parameters())
    for m in model.modules():
        if isinstance(m, bnb.nn.Linear8bitLt):
            s = getattr(m.weight, "SCB", None)
            s = s if s is not None else getattr(m.state, "SCB", None)
            if s is not None:
                total += s.numel() * s.element_size()
    return total


def param_checksum(model):
    return float(sum(p.detach().double().sum().item() for p in model.parameters()))


CONFIGS = {
    "baseline": dict(name="FP32 baseline Transformer", precision="FP32", ckpt="baseline_seed0.pt", build=build_baseline),
    "linear": dict(name="FP32 linear-attention variant", precision="FP32", ckpt="linear_seed0.pt", build=build_linear),
    "int8": dict(name="INT8 hidden layers + FP32 head (bitsandbytes)",
                 precision="INT8 (bitsandbytes LLM.int8) on 16 hidden Linear layers; FP32 128->6 head, embeddings, LayerNorm",
                 ckpt="baseline_seed0.pt", build=build_baseline),
    "student": dict(name="FP32 distilled student", precision="FP32", ckpt="student_seed0.pt", build=build_student),
}


def load_model(cfg):
    m = cfg["build"]()
    m.load_state_dict(torch.load(f"{OUT_DIR}/{cfg['ckpt']}", map_location="cpu"))
    info = {}
    if cfg is CONFIGS["int8"]:
        done, kept = quantize_linears(m)        # validated recipe: baseline weights -> INT8 hidden layers, head FP32
        info = {"layers_quantized": done, "layers_fp32": kept}
    return m.to(DEVICE).eval(), info


def run_benchmark(key, run_id):
    cfg = CONFIGS[key]
    free_gpu()                                   # previous model is already deleted by the caller
    background_mb = torch.cuda.memory_allocated() / MB
    model, info = load_model(cfg)
    inputs = [X["test"][i:i + BENCH_BATCH].to(DEVICE) for i in BENCH_IDX]    # not timed
    free_gpu()

    # --- confirmations before timing ---
    assert not model.training, "model must be in eval mode"
    assert all(p.device.type == DEVICE.type for p in model.parameters()), "model parameters are not on CUDA"
    assert all(t.device.type == DEVICE.type and tuple(t.shape) == (BENCH_BATCH, MAX_LEN) for t in inputs), "bad input"
    pc = param_counts(model)
    int8_info = None
    if key == "int8":
        q_mods = [m for m in model.modules() if isinstance(m, Int8Linear)]
        fp_lin = [m for m in model.modules() if type(m) is nn.Linear]       # plain FP32 Linear layers only
        w_q = sum(m.q.weight.numel() for m in q_mods)
        w_fp = sum(m.weight.numel() for m in fp_lin)
        ok = (len(q_mods) == 16 and len(fp_lin) == 1 and fp_lin[0] is model.head
              and all(m.q.weight.dtype == torch.int8 and m.q.weight.device.type == DEVICE.type for m in q_mods)
              and model.head.weight.dtype == torch.float32 and model.head.weight.device.type == DEVICE.type)
        int8_info = dict(n_quantized=len(q_mods), n_fp32=len(fp_lin), weights_quantized=w_q, weights_fp32=w_fp,
                         pct_quantized=100 * w_q / (w_q + w_fp), verified=ok)
        print(f"  INT8 check: {len(q_mods)} quantized Linear layers, {len(fp_lin)} FP32 Linear layer(s), "
              f"{int8_info['pct_quantized']:.4f}% of Linear weights quantized -> {'OK' if ok else 'NOT OK'}")
        if not ok:
            bench_stop("the INT8 model is not 16 bitsandbytes INT8 hidden layers + FP32 head on CUDA", json.dumps(int8_info))
    checksum_pre = param_checksum(model)

    # --- memory + latency (the measured region) ---
    torch.cuda.reset_peak_memory_stats()
    resident = torch.cuda.memory_allocated()
    smi_before = gpu_state()
    try:
        with torch.no_grad():
            grad_enabled_inside = torch.is_grad_enabled()
            for w in range(N_WARMUP):                                    # untimed warm-up
                out = model(inputs[w % N_DOCS])
            if out.device.type != DEVICE.type:
                bench_stop("the model output is not on CUDA", f"output device: {out.device}")
            cuda_sync()
            # bitsandbytes casts INT8-layer biases to FP16 on the first forward (one-time, not training), so the
            # reference checksum for 'parameters unchanged during timing' is taken after warm-up.
            checksum_warm = param_checksum(model)
            syncs_start = SYNC_COUNT
            lat, positions = [], []
            for t in range(N_TRIALS):
                pos = t % N_DOCS
                cuda_sync()                                              # before starting the measured inference
                t0 = time.perf_counter()
                out = model(inputs[pos])                                 # exactly one forward pass
                cuda_sync()                                              # after the forward, before stopping the timer
                lat.append((time.perf_counter() - t0) * 1000.0)
                positions.append(pos)
            timed_syncs = SYNC_COUNT - syncs_start
    except RuntimeError:
        raise
    except Exception as e:
        bench_stop(f"inference failed for {cfg['name']}", f"{type(e).__name__}: {str(e)[:300]}")
    peak = torch.cuda.max_memory_allocated()
    smi_after = gpu_state()

    # --- after measurement: weight storage, state checks, fidelity to the recorded seed-0 accuracy ---
    storage = weight_bytes(model)
    checksum_after = param_checksum(model)
    state_match = None
    if key == "int8":
        saved = torch.load(f"{OUT_DIR}/int8_mixed_seed0.pt", map_location="cpu")
        now = {k: v for k, v in model.state_dict().items()}
        missing_keys = [k for k in saved if k not in now]
        mismatched = []
        for k in saved:
            if k in now:
                a, b = saved[k], now[k]
                same = torch.equal(a.cpu(), b.detach().cpu()) if torch.is_tensor(a) else (a == b)
                if not same:
                    mismatched.append(k)
        state_match = {"saved_tensors": len(saved), "missing": missing_keys, "mismatched": mismatched}
    test_acc = float((predict(model, X["test"]) == Y["test"].numpy()).mean())   # untimed fidelity check
    inputs_sha = hashlib.sha256(b"".join(t.cpu().numpy().tobytes() for t in inputs)).hexdigest()[:16]
    result = dict(key=key, run=run_id, name=cfg["name"], precision=cfg["precision"], params=pc, latencies_ms=lat,
                  doc_positions=positions, test_indices=[BENCH_IDX[p] for p in positions],
                  p50=float(np.percentile(lat, 50)), p95=float(np.percentile(lat, 95)), mean=float(np.mean(lat)),
                  min=float(np.min(lat)), max=float(np.max(lat)),
                  resident_mb=resident / MB, peak_mb=peak / MB, incremental_mb=(peak - resident) / MB,
                  background_mb=background_mb, weight_storage_mb=storage / MB, weight_storage_bytes=storage,
                  n_warmup=N_WARMUP, n_trials=len(lat), timed_syncs=timed_syncs, grad_enabled_inside=grad_enabled_inside,
                  out_requires_grad=bool(out.requires_grad), any_param_grad=any(p.grad is not None for p in model.parameters()),
                  params_unchanged=(checksum_warm == checksum_after) and (key == "int8" or checksum_pre == checksum_after), model_training=model.training,
                  input_shape=list(inputs[0].shape), inputs_sha=inputs_sha, gpu=BENCH_ENV["gpu_name"], pid=os.getpid(),
                  int8=int8_info, int8_state_match=state_match, full_test_acc=test_acc, gpu_state_before=smi_before,
                  gpu_state_after=smi_after, vocab_in_model=model.tok_emb.num_embeddings)
    print(f"  {cfg['name']} (run {run_id}): p50 {result['p50']:.3f} ms | p95 {result['p95']:.3f} ms | mean {result['mean']:.3f} ms | "
          f"resident {result['resident_mb']:.2f} MB | peak {result['peak_mb']:.2f} MB | GPU state {smi_before}")
    del model, inputs, out
    free_gpu()
    return result

In [ ]:
# --- Run: baseline (run 1), linear, INT8, student, then baseline again for the stability check -------------
BENCH_ORDER = [("baseline", 1), ("linear", 1), ("int8", 1), ("student", 1), ("baseline", 2)]
BENCH_RUNS = []
for _key, _run in BENCH_ORDER:
    print(f"Benchmarking {CONFIGS[_key]['name']} (run {_run}) ...")
    BENCH_RUNS.append(run_benchmark(_key, _run))
TEST_TENSOR_SHA_AFTER = hashlib.sha256(X["test"].numpy().tobytes()).hexdigest()[:16]
print("\nAll benchmark runs finished. Gradients were never enabled and no model was trained.")

## Benchmark results (batch size 1, sequence length 128, seed-0 models; quality from three seeds)
Memory is PyTorch allocator memory (MB = 1,048,576 bytes), not the CUDA context or driver memory.
This table is a measurement, not a ranking.

In [ ]:
import pandas as pd


def _load(name):
    with open(f"{OUT_DIR}/{name}") as f:
        return json.load(f)


J = {"baseline": _load("baseline_results_cuda.json")["baseline"],
     "linear": _load("linear_results_cuda.json")["linear"],
     "int8": _load("int8_mixed_results_cuda.json")["int8_mixed"],
     "student": _load("student_results_cuda.json")["student"]}


def quality(key, field):
    v = np.array([J[key][f"seed{s}"][field] for s in SEEDS])
    return float(v.mean()), float(v.std(ddof=1))


RUN1 = {r["key"]: r for r in BENCH_RUNS if r["run"] == 1}
rows = []
for key in ["baseline", "linear", "int8", "student"]:
    r = RUN1[key]
    am, asd = quality(key, "overall_acc")
    rm, rsd = quality(key, "rare_class_recall_macro")
    rows.append({"Configuration": r["name"], "Precision / quantization": r["precision"],
                 "Total parameters": r["params"]["total"], "Non-embedding parameters": r["params"]["non_embedding"],
                 "Overall accuracy mean (3 seeds)": am, "Overall accuracy std (3 seeds)": asd,
                 "Rare-class macro recall mean (3 seeds)": rm, "Rare-class macro recall std (3 seeds)": rsd,
                 "p50 latency ms": r["p50"], "p95 latency ms": r["p95"], "Mean latency ms": r["mean"],
                 "Peak GPU allocated memory MB": r["peak_mb"], "Resident GPU allocated memory MB": r["resident_mb"],
                 "Model weight storage MB": r["weight_storage_mb"]})
summary = pd.DataFrame(rows)
_short = {"FP32 baseline Transformer": "FP32 baseline", "FP32 linear-attention variant": "FP32 linear-attn",
          "INT8 hidden layers + FP32 head (bitsandbytes)": "INT8 hidden + FP32 head", "FP32 distilled student": "FP32 student"}
def _fmt(col, v):
    if "parameters" in col: return f"{int(v):,}"
    if "accuracy" in col or "recall" in col: return f"{v:.4f}"
    if "latency" in col: return f"{v:.3f}"
    return f"{v:.3f}"          # MB


_view = summary.drop(columns=["Precision / quantization"]).set_index("Configuration")
_view.index = [_short[i_] for i_ in _view.index]
_disp = pd.DataFrame({c_: [_fmt(c_, v_) for v_ in _view[c_]] for c_ in _view.columns}, index=_view.index).T
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(_disp.to_string())
print("\nPrecision / quantization:")
for _, r_ in summary.iterrows():
    print(f"  {_short[r_['Configuration']]}: {r_['Precision / quantization']}")

print("\nAdditional latency and memory detail (run 1, ms and MB)")
detail = pd.DataFrame([{"Configuration": RUN1[k]["name"], "min ms": RUN1[k]["min"], "p50 ms": RUN1[k]["p50"],
                        "mean ms": RUN1[k]["mean"], "p95 ms": RUN1[k]["p95"], "max ms": RUN1[k]["max"],
                        "resident MB": RUN1[k]["resident_mb"], "peak MB": RUN1[k]["peak_mb"],
                        "incremental peak MB": RUN1[k]["incremental_mb"]} for k in RUN1])
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.float_format", "{:.4f}".format):
    print(detail.to_string(index=False))
_bg = ", ".join("%.3f" % r_["background_mb"] for r_ in BENCH_RUNS)
print("\nBackground allocated GPU memory before loading each model: " + _bg + " MB (leftover small tensors from earlier cells, not part of any model)")
i8 = RUN1["int8"]["int8"]
print(f"INT8 model: {i8['n_quantized']} quantized Linear layers, {i8['n_fp32']} FP32 Linear layer (the 128->6 head), "
      f"{i8['pct_quantized']:.4f}% of eligible Linear weights quantized")
print("Weight storage for INT8 is measured after the forward passes (bitsandbytes casts INT8-layer biases to FP16 then).")
print("\nMethodology note: quality columns use the existing three-seed results (seeds 0, 1, 2), not seed-0 alone. Latency and")
print("memory use the seed-0 models only. Across seeds the architecture, tensor shapes, precision and execution path are unchanged,")
print("so repeated latency trials on one representative trained seed are used while seed-to-seed variation is measured for quality.")

# --- Baseline repeatability check -----------------------------------------------------------------------------
b1 = next(r for r in BENCH_RUNS if r["key"] == "baseline" and r["run"] == 1)
b2 = next(r for r in BENCH_RUNS if r["key"] == "baseline" and r["run"] == 2)
d50 = 100 * abs(b2["p50"] - b1["p50"]) / b1["p50"]
d95 = 100 * abs(b2["p95"] - b1["p95"]) / b1["p95"]
UNSTABLE = d50 > 15 or d95 > 15
print("\n--- Baseline repeatability check (stability check only; run 1 is the benchmark value) ---")
print(f"{'':<14}{'p50 ms':>10}{'p95 ms':>10}")
print(f"{'run 1':<14}{b1['p50']:>10.3f}{b1['p95']:>10.3f}")
print(f"{'run 2':<14}{b2['p50']:>10.3f}{b2['p95']:>10.3f}")
print(f"{'difference':<14}{d50:>9.1f}%{d95:>9.1f}%   (|run2 - run1| / run1; threshold 15%)")
print("GPU state (SM clock, max SM clock, temp, power) before/after run 1:", b1["gpu_state_before"], "|", b1["gpu_state_after"])
print("GPU state before/after run 2:", b2["gpu_state_before"], "|", b2["gpu_state_after"])

# --- Save --------------------------------------------------------------------------------------------------
raw = pd.DataFrame([{"configuration": r["name"], "run": r["run"], "trial": t + 1, "test_example_index": r["test_indices"][t],
                     "document_position": r["doc_positions"][t], "latency_ms": r["latencies_ms"][t]}
                    for r in BENCH_RUNS for t in range(len(r["latencies_ms"]))])
raw.to_csv(f"{OUT_DIR}/benchmark_raw_latency.csv", index=False)
summary.to_csv(f"{OUT_DIR}/benchmark_summary.csv", index=False)
with open(f"{OUT_DIR}/benchmark_meta.json", "w") as f:
    json.dump({"env": BENCH_ENV, "workload": {"batch": BENCH_BATCH, "seq_len": MAX_LEN, "n_docs": N_DOCS, "warmup": N_WARMUP,
               "trials": N_TRIALS, "index_seed": INDEX_SEED, "index_sha256": BENCH_IDX_SHA, "indices": BENCH_IDX},
               "dataset_sha256": DATA_SHA256, "repeatability": {"p50_pct_diff": d50, "p95_pct_diff": d95, "unstable": UNSTABLE},
               "int8": i8, "runs": [{k: v for k, v in r.items() if k not in ("latencies_ms", "doc_positions", "test_indices")}
                                    for r in BENCH_RUNS]}, f, indent=1, default=str)
print(f"\nsaved: {OUT_DIR}/benchmark_raw_latency.csv ({len(raw)} rows), {OUT_DIR}/benchmark_summary.csv, {OUT_DIR}/benchmark_meta.json")

if UNSTABLE:
    print("\n" + "!" * 100)
    print("GPU TIMING ENVIRONMENT FLAGGED UNSTABLE: baseline p50 or p95 differs by more than 15% between two runs.")
    print("The latency comparison above must NOT be treated as final. Nothing was re-run to get a better result.")
    print("!" * 100)
else:
    print("\nBaseline repeatability within 15%: the timing environment is stable enough to treat this comparison as final.")

## Benchmark verification checks

In [ ]:
bk = []
def bchk(name, ok, detail=""): bk.append((name, bool(ok), detail))

allr = BENCH_RUNS
hashes = {}
for jn, fn in (("baseline", "baseline_results_cuda.json"), ("linear", "linear_results_cuda.json"), ("student", "student_results_cuda.json")):
    hashes[jn] = _load(fn)["config"]["data_sha256"]
bchk("All four configurations use the same dataset fingerprint (reference, baseline, linear, student runs; INT8 rebuilt from baseline)",
     len(set(hashes.values())) == 1 and list(hashes.values())[0] == DATA_SHA256 == REFERENCE_DATA_SHA256, DATA_SHA256[:16] + "...")
bchk("Same vocabulary: every model's embedding table has the notebook's vocab size, and the saved runs recorded the same vocab",
     all(r["vocab_in_model"] == VOCAB for r in allr)
     and all(_load(fn)["config"]["vocab"] == VOCAB for fn in ("baseline_results_cuda.json", "linear_results_cuda.json", "student_results_cuda.json")),
     f"vocab {VOCAB}")
bchk("Every model received 1 x 128 input tensors", all(r["input_shape"] == [BENCH_BATCH, MAX_LEN] for r in allr), str(allr[0]["input_shape"]))
bchk("All used the exact same 100 test-document indices and order (identical input tensors in every run)",
     len({r["inputs_sha"] for r in allr}) == 1 and all(r["test_indices"] == allr[0]["test_indices"] for r in allr)
     and sorted(set(allr[0]["test_indices"])) == sorted(BENCH_IDX), f"index sha {BENCH_IDX_SHA}, input sha {allr[0]['inputs_sha']}")
bchk("All runs on the same Tesla T4 in the same session/process", len({r["gpu"] for r in allr}) == 1 and len({r["pid"] for r in allr}) == 1
     and "T4" in allr[0]["gpu"], f"{allr[0]['gpu']}, pid {allr[0]['pid']}")
bchk("15 warm-up iterations in every run", all(r["n_warmup"] == 15 for r in allr))
bchk("100 measured trials in every run, 100 raw latencies each", all(r["n_trials"] == 100 and len(r["latencies_ms"]) == 100 for r in allr))
bchk("CUDA synchronize before and after every timed forward (2 per trial)", all(r["timed_syncs"] == 2 * N_TRIALS for r in allr),
     f"{allr[0]['timed_syncs']} syncs per run")
bchk("No model trained during benchmarking (parameters bit-identical before and after, no .grad, eval mode)",
     all(r["params_unchanged"] and not r["any_param_grad"] and not r["model_training"] for r in allr))
bchk("No gradients enabled (torch.no_grad active, outputs carry no graph)",
     all((not r["grad_enabled_inside"]) and (not r["out_requires_grad"]) for r in allr))
bchk("Test examples unchanged across the whole benchmark (test tensor hash identical before and after)",
     TEST_TENSOR_SHA_BEFORE == TEST_TENSOR_SHA_AFTER, TEST_TENSOR_SHA_BEFORE)
bchk("Each rebuilt model reproduces its recorded seed-0 test accuracy (within one test document)",
     all(abs(r["full_test_acc"] - J[r["key"]]["seed0"]["overall_acc"]) <= 1 / len(Y["test"]) + 1e-9 for r in allr),
     ", ".join(f"{r['key']}{r['run']}: {r['full_test_acc']:.4f} vs {J[r['key']]['seed0']['overall_acc']:.4f}" for r in allr))
sm = RUN1["int8"]["int8_state_match"]
bchk("Rebuilt INT8 model matches the saved int8_mixed_seed0.pt (all saved tensors equal)",
     sm["missing"] == [] and sm["mismatched"] == [], f"{sm['saved_tensors']} saved tensors, missing {sm['missing']}, mismatched {sm['mismatched'][:3]}")
bchk("INT8 model: 16 bitsandbytes INT8 layers, FP32 128->6 head, int8 weights on CUDA", RUN1["int8"]["int8"]["verified"],
     f"{i8['n_quantized']} INT8, {i8['n_fp32']} FP32 Linear, {i8['pct_quantized']:.4f}% of Linear weights quantized")
bchk("Stored parameter counts match the saved quality runs",
     all(RUN1[k]["params"] == J[k]["seed0"]["params"] for k in RUN1))
bchk("Peak allocated >= resident allocated for every run", all(r["peak_mb"] >= r["resident_mb"] for r in allr))
bchk("Environment recorded (GPU, compute capability, CUDA, PyTorch, bitsandbytes, Python)", all(BENCH_ENV.values()),
     f"{BENCH_ENV['gpu_name']}, cc {BENCH_ENV['gpu_compute_capability']}, CUDA {BENCH_ENV['cuda_version']}, torch {BENCH_ENV['pytorch_version']}, "
     f"bnb {BENCH_ENV['bitsandbytes_version']}, Python {BENCH_ENV['python_version']}")
bchk("Baseline repeatability: p50 and p95 each within 15% across the two baseline runs", not UNSTABLE, f"p50 {d50:.1f}%, p95 {d95:.1f}%")
bchk("benchmark_raw_latency.csv holds every timing trial", len(raw) == 100 * len(BENCH_RUNS), f"{len(raw)} rows")
bchk("benchmark_summary.csv saved", os.path.exists(f"{OUT_DIR}/benchmark_summary.csv"))

print(f"{'check':<132} result")
for n, ok, dtl in bk:
    print(f"{n:<132} {'PASS' if ok else 'FAIL'}  {dtl}")
if all(ok for _, ok, _ in bk):
    print("\nAll benchmark verification checks passed.")
else:
    print("\nSome benchmark checks FAILED: do not treat the comparison as final.")
    raise AssertionError("benchmark verification failed")